# Experimental validation of the FLOW-3D WELD laser-melting model

This notebook is the **reproducibility supplementary** of the simulation
pipeline - the validation section of the manuscript in executable form. The
complete input pipeline (all campaign configs, frozen material files, the
pre-registered plan and outcomes) is tracked in `flow3D_scripts/validation/`;
`validation/reproduce.py` re-renders every case without a license or re-solves
the campaign with one, and `F3D_RUNS_ROOT` points this notebook at any of the
three data routes (workstation tree / fresh solves / published NPZ archive).
As a validation section: every figure
compares a simulation against **published experimental measurements**, using data
downloaded from the original repositories rather than numbers copied out of
papers. The model's physical constants come from each material's property files via
the FLOW-3D documentation's own recipes, with one exception: the Ti64
saturation anchor **tv1 = 3520 +- ~95 K is a calibrated scalar** (fitted on the
Gan 129 W condition, clipped to the literature bracket), and the IN718
component drops were selected by agreement with the 285 W base condition. That
calibration is disclosed inline where each affected figure appears rather than
presented as prediction.

## Data sources

| source | DOI / ref | what we use | how shared |
|---|---|---|---|
| NIST AM Bench **AMB2022-03** | [10.18434/mds2-2718](https://doi.org/10.18434/mds2-2718) | IN718 bare-plate single-track melt-pool cross-sections (depth, width) over a P/V/spot matrix | public repository, 233 individual measurements |
| NIST **A-AMB2022-01** | [10.18434/mds2-2525](https://doi.org/10.18434/mds2-2525) | Ti-6Al-4V bare plate: calibrated absorptance trace (+/-1.3 %) and 50 kfps X-ray radiographs of the vapour depression | public repository |
| **Gan et al. 2021** | [Nat Commun 12:2379](https://doi.org/10.1038/s41467-021-22704-0) | keyhole depth vs (P, V, spot) for Ti64 / SS316 — "directly quantified from in situ X-ray images" (their Methods); the sheet's eta column is model-derived and unused here | open-access Supplementary Data 1 |
| **Simonds et al. 2021** | [Appl Mater Today 23:101049](https://doi.org/10.1016/j.apmt.2021.101049) | stationary-spot onsets: vapour depression ~96 W, keyhole ~120 W (2 ms, Ti64) | values stated in text |

Run `python fetch_experimental_data.py` once to populate `../../data/experimental/`
(gitignored). Simulation results are read from the FLOW-3D batch directories.

## What is being validated

1. **Absorbed power** - the model must couple the right fraction of the beam into
   the metal, including the enhancement from multiple reflections inside the
   cavity. This is the observable that exposed the historical calibration error.
2. **Melt-pool geometry** - absolute depth and width against metallography, and
   the *trends* with power, speed and spot size.
3. **Keyhole depth** - vapour-depression depth against X-ray measurements, and
   the conduction -> keyhole transition.

**Agreement band, used everywhere:** within +-20 % of the measurement or within
2 s.d. of its scatter (the +-20 % clause governs in practice; the NIST n=6
scatter is 1-3 % of target). All simulated lengths quantise to that case's cell size (mostly 10 um, but 6, 25 and 50 um cases exist -- it is read per case, never assumed),
so a leg whose value sits within half a cell of a band edge is reported as
indeterminate rather than counted.

In [ ]:
import io, json, glob, pathlib, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 200, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "figure.autolayout": True,
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10, "legend.fontsize": 9,
})

EXP = Path("../../data/experimental")
# Run root: F3D_RUNS_ROOT may point at (a) the workstation batch tree, (b) a
# tree produced by `flow3D_scripts/validation/reproduce.py --solve`, or (c) an
# unpacked archive of the published solved NPZs (layout <root>/<case>/npz/).
import os
RUNS = Path(os.environ.get("F3D_RUNS_ROOT",
                           str(Path.cwd().resolve().parents[2] / "flow3D_scripts" / "runs" / "batches")))
SIM, EXPC, ACC = "#4c72b0", "#c44e52", "#55a868"   # simulation / experiment / accent
# NOTE: there is deliberately no CELL_UM constant. Cell size is read per case
# via cell_um(z) -- 14 of 96 campaign cases ran at 6, 25 or 50 um, so a fixed
# 10.0 mis-scaled every length they reported by up to 5x.

print("experimental data:", EXP.resolve())
print("  present:", sorted(p.name for p in EXP.glob("*")) if EXP.exists()
      else "MISSING -> run fetch_experimental_data.py")

In [ ]:
# ----------------------------------------------------------------- helpers --
# The metric definitions live in ONE place: flow3D_scripts/python/melt_metrics.py,
# which is tested (tests/test_melt_metrics.py) and documented in
# flow3D_scripts/validation/ACCEPTANCE.md. This notebook deliberately does NOT
# re-derive them. Every geometric metric in the calibration campaign used to be
# copy-pasted per script; the copies drifted, and the drift produced retracted
# results -- an aspect ratio taking width from a different frame than depth, four
# comparisons of a simulated vapour depression against a metallographic fusion
# depth, and a hardcoded 10 um cell size applied to cases that ran at 6, 25 and
# 50 um (wrong by up to 5x).
import sys

SCRIPTS = Path(os.environ.get("F3D_SCRIPTS_ROOT", str(Path.cwd().resolve().parents[2] / "flow3D_scripts")))
sys.path.insert(0, str(SCRIPTS / "python"))
import melt_metrics as mm
import score_anchors as sa

print("metrics from:", (SCRIPTS / "python" / "melt_metrics.py"))
print("  acceptance band:", mm.DEFAULT_BAND)


# Source of truth for solved cases. AiiDA first when a profile is reachable, so the
# notebook runs from provenance-tracked, presolved simulations rather than from a
# local scratch tree; the run directory is the fallback for cases not yet ingested.
# The ingested payload is the FULL export (same layout, all fields), so the metrics
# below are identical either way -- verified case-by-case, not assumed.
AIIDA_PROFILE = os.environ.get("F3D_AIIDA_PROFILE")
_AIIDA_CACHE = {}


def _aiida_index():
    """{case_name: node_uuid} for every ingested FLOW-3D simulation, or {}."""
    if _AIIDA_CACHE:
        return _AIIDA_CACHE
    try:
        from aiida import load_profile
        from aiida.orm import QueryBuilder, Node
        load_profile(AIIDA_PROFILE)
        # Project the case name itself, never the whole simulation_metadata blob.
        # Measured on this profile (4471 simulations): projecting the blob and
        # reading .case_name in Python takes 12.3 s, the leaf path
        # attributes.simulation_metadata.case_name 0.8 s, and extras.case_name
        # 0.6 s -- so the cost was pulling thousands of large JSON documents, not
        # which container they live in.
        #
        # extras is also STRICTLY better covered, which is a correctness point
        # rather than a speed one: extras.case_name returns 4471 rows against the
        # attributes path's 4062, with 409 extras-only and ZERO attributes-only.
        # This index used to miss those 409, so their cases silently fell through
        # to the run-tree fallback and read as "not exported yet" when absent.
        qb = QueryBuilder().append(
            Node, filters={"node_type": {"like": "%PinoSimulationData%"}},
            project=["uuid", "extras.case_name"])
        for uuid, name in qb.all():
            if name:
                _AIIDA_CACHE[str(name)] = uuid
    except Exception as exc:                      # no profile, no daemon, plugin clash
        print(f"  [aiida] unavailable ({type(exc).__name__}); using the run tree")
        _AIIDA_CACHE["__unavailable__"] = True
    return _AIIDA_CACHE


def load_aiida(case):
    """Exported arrays for a case straight out of the AiiDA repository, or None."""
    idx = _aiida_index()
    uuid = idx.get(case)
    if not uuid:
        return None
    from aiida.orm import load_node
    node = load_node(uuid)
    with node.base.repository.open("payload.npz", "rb") as fh:
        return np.load(io.BytesIO(fh.read()), allow_pickle=True)


def load_npz(batch, case, prefer_aiida=True):
    # A case's exported arrays, or None if it has not finished yet.
    # Order: AiiDA (provenance-tracked) -> workstation batch tree -> flat archive.
    if prefer_aiida:
        z = load_aiida(case)
        if z is not None:
            return z
    for pat in (RUNS / batch / "runs" / case / "npz" / "*.npz",
                RUNS / case / "npz" / "*.npz"):
        hits = sorted(glob.glob(str(pat)))
        if hits:
            return np.load(hits[0])
    return None


def cell_um(z):
    # This case's cell size -- the resolution floor on every length below.
    # Read per case, never assumed: see mm.cell_size_um.
    return mm.cell_size_um(z)


def depression(z, frame=None):
    # Steady-window vapour-depression depth/width/sd (um), or one frame's
    # geometry when frame= is given (imagery only -- a single frame sits inside
    # the +-30-42 um oscillation amplitude).
    if frame is not None:
        g = mm.depression_frame(z, frame)
        return g.depth_um, g.width_um, frame
    d = mm.depression(z)
    return d.depth_um, d.width_um, d.depth_sd_um


def depression_series(z):
    # Vapour-depression depth (um) for every stored frame.
    return mm.depression_series(z)


def fusion_zone(z, half_window=5):
    # Melt-pool depth/width (um) from the ever-melted region, sectioned AT THE
    # BEAM. Metallography sections a solidified track, so the fusion zone is the
    # union over time rather than a single frame; `fusion_geometry_auto` chooses
    # the plane perpendicular to the beam's own travel and falls back to the
    # mid-track slice when a run carries no deposition field to locate it with.
    # The scorer moved to the same call on 2026-08-05, so notebook and scorer
    # cannot report different numbers for one case.
    # `fusion_geometry_auto` sections at the beam and is what the scorer calls.
    # Older checkouts of flow3D_scripts do not have it, so fall back rather than
    # crash; the printout in the helpers cell says which one is in use, because
    # the two differ by up to a cell in width on straight tracks and by far more
    # on random paths.
    _fz = getattr(mm, "fusion_geometry_auto", None)
    g = _fz(z) if _fz else mm.fusion_geometry(z, track_half_cells=half_window)
    return g.depth_um, g.width_um


def absorbed_fraction(batch, case, power_w, t0=2e-4, t1=8e-4):
    # Net absorbed fraction from the solver energy audit over a quiet window.
    # NET, so it reads lower than a measured GROSS absorptance -- see mm's note.
    hits = glob.glob(str(RUNS / batch / "runs" / case / "hd3out.*"))
    return mm.absorbed_fraction(hits[0], power_w, t0, t1) if hits else np.nan


def band(sim, exp, sd, rel=0.20):
    # Agreement: within +/-rel of the measurement, or within 2 sd of it.
    # rel=0.20 is deliberately STRICTER than the documented band (mm.DEFAULT_BAND
    # = 0.30, derived from the sources' own scatter). Kept strict so that nothing
    # this notebook calls a pass could fail the documented criterion; the assert
    # stops it from ever being loosened past that criterion by accident.
    assert rel <= mm.DEFAULT_BAND, (
        f"rel={rel} is looser than the documented band {mm.DEFAULT_BAND}; "
        "loosening an acceptance criterion after seeing results is method rule 2")
    return abs(sim - exp) <= max(rel * exp, 2 * sd)


def laser_off_s_of(batch, case):
    """When the simulated beam stops being trustworthy, or None if it never does.

    Two things can end the useful part of a run, and neither is enforced by the
    rendered input: the beam is never switched off (single-row `powlbm`/`utlbm`,
    which FLOW-3D holds at the last value), so it keeps firing and keeps moving to
    the finish time.

      1. The config asked for a turn-off. `derived.laser_off_time_s` records the
         intent even though it was never rendered, so that is the cut.
      2. The beam reaches the edge of the domain. Heating piles up against the
         boundary there, which is a mesh artifact rather than physics, so the cut
         is when the beam centre comes within two beam radii of the x boundary.

    Whichever comes first wins. A run that neither declares a turn-off nor drives
    the beam near the boundary -- a stationary dwell in a large domain, say -- has
    nothing to cut, and this returns None so the full series is kept.
    """
    man = None
    for p in (RUNS / batch / "runs" / case / "case_manifest.json",
              RUNS / case / "case_manifest.json"):
        try:
            man = json.loads(p.read_text()); break
        except Exception:
            continue
    if man is None:
        return None
    ends = []
    off = (man.get("derived") or {}).get("laser_off_time_s")
    if off:
        ends.append(float(off))
    las = (man.get("input") or {}).get("laser") or {}
    v = float(las.get("scan_speed_m_per_s") or 0.0)
    start = (las.get("start_point_m") or [None])[0]
    r0 = float(las.get("beam_radius_m") or 0.0)
    if v > 0 and start is not None:
        try:
            x_max = float(np.load(RUNS_NPZ_CACHE[case])["x"].max()) if False else None
        except Exception:
            x_max = None
        geo = (man.get("input") or {}).get("geometry") or {}
        length = geo.get("length_m")
        if length:
            x_max = float(length) / 2.0
        if x_max is not None:
            t_edge = (x_max - 2.0 * r0 - float(start)) / v
            if t_edge > 0:
                ends.append(t_edge)
    return min(ends) if ends else None


def cut_at_off(z, batch, case):
    """The arrays truncated to the trustworthy part of the run (see above)."""
    off = laser_off_s_of(batch, case)
    if z is None or off is None:
        return z
    t = z["time_s"]; k = int((t <= off).sum())
    if k < 2 or k >= len(t):
        return z
    return {n: (a[:k] if getattr(a, "ndim", 0) >= 1 and a.shape[0] == len(t) else a)
            for n, a in z.items()}


print("fusion metric:", "at-beam (fusion_geometry_auto)"
      if hasattr(mm, "fusion_geometry_auto") else "mid-track (legacy)")
print("helpers ready")


---
## 1. Absorbed power - Ti-6Al-4V, NIST A-AMB2022-01

The most diagnostic single quantity. A flat Ti64 surface at 1070 nm absorbs only
~35 % of the beam at normal incidence; a keyhole traps light through multiple
reflections and pushes the absorbed fraction far higher. Reproducing the measured
value therefore tests the absorption model *and* the cavity shape at once.

The simulation runs at the published conditions (195 W mean input, 0.7 m/s,
122.5 um 1/e^2 spot => 61.25 um radius, bare 300 um plate) with the final Ti64
recipe (tv1 = 3520 K, no plume). The measurement is the calibrated
integrating-sphere trace shipped with the dataset.

**Methodology note.** The integrating sphere measures *gross* optical
absorption; the solver's energy audit reports thermal energy *retained* in the
fluid. The hd3out mass audit shows evaporative carry-off < 0.1 % of beam power
here, so the two differ mainly by ~1-2 points of conduction losses.

**Reading the trace.** The measurement is not flat, and the shape is physics
rather than noise. It starts near **29 %** on the cold solid surface, close to the
~35 % a flat Ti64 face absorbs at normal incidence, climbs through melting, and
peaks at **73.5 % at 0.228 ms** before settling to a **48.5 %** plateau. The peak
is the cavity: while the beam is still sinking a deep, near-vertical capillary it
traps light through many internal reflections. Once translation carries the beam
about a spot diameter (2*r0/V = 175 us) the keyhole takes up its shallower,
rear-inclined steady shape, fewer reflections occur, and the absorbed fraction
falls. The plateau is the quantity to compare against a steady simulation, which
is why the averaging window starts after the transient and not at laser-on.

**Scope note.** This 61.25 um spot lies outside the validated scope (33.5-
47.5 um anchors). It is reported as the out-of-scope diagnostic it is: the
final recipe lands closest of any arm tested (net 44.3 % vs 48.5 % measured (the measured baseline is **48.5 %**, not the 50.8 % quoted before: the old window started 0.2 ms after laser-on, inside the drilling transient that peaks at 73.5 % near 0.23 ms. From 0.45 ms the value is flat against the choice, 48.41-48.59 % for starts of 0.40-0.70 ms);
the retired plume arm read 39.8 %, the old over-volatile no-plume arm
58.9-60.8 %), while the depth remains below the X-ray band. Not a pass - a
documented boundary that the re-anchor moved in the right direction.

In [ ]:
abs_csv = EXP / "nist_ti64_scan_absorption.csv"
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [2, 1]})

meas_qs = np.nan
if abs_csv.exists():
    d = pd.read_csv(abs_csv)
    t = d["Time"].to_numpy() * 1e3                 # ms
    pin = d["InputLaser"].to_numpy()
    pab = d["AbsoluteAbsorption"].to_numpy()
    on = pin > 50
    t_on = t[on]
    frac_t = np.where(on, pab / np.maximum(pin, 1e-9), np.nan)
    ax[0].plot(t, frac_t * 100, lw=0.6, color=EXPC, alpha=0.85,
               label="measured (NIST, 40 ns samples)")
    # The trace has a large drilling transient before it settles: 29 % on the
    # cold solid surface (close to the 35 % flat-surface Fresnel value), rising
    # through melting to a 73.5 % peak at 0.228 ms, then decaying to a plateau.
    # The peak is the cavity itself -- while the beam is still sinking a deep,
    # near-vertical capillary it traps light through many internal reflections.
    # Once translation carries the beam a spot diameter (2*r0/V = 175 us) the
    # keyhole takes up its shallower rear-inclined steady shape, fewer
    # reflections occur, and the absorbed fraction drops.
    #
    # The window therefore has to START AFTER that transient. It began at
    # t_on.min() + 0.2 ms, which is inside it, and that pulled the "quasi-steady"
    # value up to 50.8 %. Measured from 0.45 ms the value is 48.5 % and is flat
    # against the choice: 48.59 / 48.47 / 48.49 / 48.41 / 48.53 % for starts of
    # 0.40 / 0.45 / 0.50 / 0.60 / 0.70 ms, a spread of 0.2 pt. Starting at 0.20
    # or 0.30 ms instead gives 50.83 % and 49.61 %, i.e. only the windows that
    # overlap the transient disagree.
    QS_START_MS = 0.45
    qs = on & (t > QS_START_MS) & (t < t_on.max() - 0.1)
    meas_qs = pab[qs].sum() / pin[qs].sum()
    peak_i = int(np.nanargmax(np.where(on & (t > 0) & (t < 0.5), frac_t, np.nan)))
    ax[0].axhline(meas_qs * 100, color=EXPC, ls="--", lw=1.2,
                  label="measured quasi-steady %.1f %% (from %.2f ms)" % (meas_qs * 100, QS_START_MS))
    ax[0].set_xlim(t_on.min() - 0.15, t_on.max() + 0.15)

# Net absorbed fraction from the solver's fluid-energy audit. The hd3out mass
# audit shows evaporative carry-off < 0.1 % of beam power at this condition, so
# gross optical absorption ~= net retained + ~1-2 pts of conduction losses to
# the unmodelled far field. (An earlier revision added a "carry-off" term from
# the NPZ fluid-volume deficit; the mass audit is ~200x smaller -- the volume
# deficit is not evaporation -- so that term is dropped.)
NIST_BATCH, NIST_CASE, NIST_P = "bundle_gate", "t2_nist195", 195.0
sim_frac = absorbed_fraction(NIST_BATCH, NIST_CASE, NIST_P)
if np.isfinite(sim_frac):
    ax[0].axhline(sim_frac * 100, color=SIM, lw=1.6,
                  label="simulation (net audit) %.1f %%" % (sim_frac * 100))
ax[0].set_xlabel("time (ms)"); ax[0].set_ylabel("absorbed fraction (%)")
ax[0].set_title("(a) absorbed power - Ti64, 195 W, 0.7 m/s")
ax[0].set_ylim(0, 100); ax[0].legend(loc="lower right")

labels = ["flat surface\n(Fresnel, normal)", "measured\n(keyhole)", "simulation\n(keyhole)"]
vals = [35.0,
        meas_qs * 100 if np.isfinite(meas_qs) else np.nan,
        sim_frac * 100 if np.isfinite(sim_frac) else np.nan]
ax[1].bar(labels, vals, color=["0.7", EXPC, SIM])
for i, v in enumerate(vals):
    if np.isfinite(v):
        ax[1].text(i, v + 1.5, "%.0f%%" % v, ha="center", fontsize=9)
ax[1].set_ylabel("absorbed fraction (%)"); ax[1].set_ylim(0, 80)
ax[1].set_title("(b) cavity enhancement")
plt.show()

if np.isfinite(sim_frac) and np.isfinite(meas_qs):
    print("simulation ~%.1f %% (net audit; carry-off < 0.1 pt)  vs  measured %.1f %%  ->  %+.1f pts"
          % (sim_frac * 100, meas_qs * 100, (sim_frac - meas_qs) * 100))
    print("Out-of-scope diagnostic (61.25 um spot): closest of any arm tested (retired plume arm")
    print("39.8 %%, old over-volatile no-plume arm 58.9-60.8 %%), still short of the measurement.")
    print("No absorptance measurement exists inside the validated 33.5-47.5 um spot range - a")
    print("stated limit: the recipe's energy input is validated in-scope only through depth.")

---
## 2. Vapour depression, against X-ray radiography

The AM-Bench release ships processed radiographs of the same scan at 50 000
frames/s, of which every 5th frame is stored. The beam looks through the plate, so
the image is a projection, not a section.

**Contrast, corrected.** An earlier version of this section said the dark region
below the surface was the vapour depression. It is the other way round. A cavity
removes material from the beam path, so it transmits *more* and appears **bright**;
the **dark** band straddling the surface is the raised rim, where displaced metal
adds path length. The reading is not just an argument from attenuation: measuring
the bright feature at 627 us gives 60 um against a simulated 60 um, which a
misidentified feature has no reason to match.

**Three things had to be fixed before any of this could be compared**, and each is
recorded in the cell that does it:

1. *Orientation.* The simulation panel was drawn upside down, so the depression
   appeared to rise out of the surface rather than cut into it.
2. *Tilt.* The sample sits at +6.50 deg in the frame. Levelling it is what makes a
   shared vertical axis mean anything, and the sign matters: rotating the wrong way
   doubles the tilt instead of removing it.
3. *Time.* Frames were being chosen as the three DEEPEST simulated depressions,
   which are not the same instants as the experimental frames, so the beam had
   travelled different distances in the two columns. Frame times now come from the
   file numbering, t = (N - 98.634) x 20 us, which reproduces the timestamps burned
   into the images to under a microsecond.

Three figures follow: the frames side by side, the two superimposed on one micron
scale, and the simulated depth history behind them.


In [ ]:
# ---------------------------------------------------------------- geometry --
# Three corrections over the first version of this figure, all of which changed
# what the comparison shows:
#
# 1. ORIENTATION. The simulation panel used origin="upper" together with an
#    extent whose bottom was zc[-1] and top zc[0], i.e. the lowest z drawn at the
#    top. Metal rendered above gas and the depression appeared to rise out of the
#    surface instead of cutting into it.
# 2. TILT. The radiographs are recorded with the sample surface at +6.50 deg
#    (measured by rotating each frame and taking the angle that maximises the
#    sharpness of the horizontal intensity step; stable to 0.25 deg on every
#    frame that contains a surface). Levelling them is what makes a shared
#    vertical axis meaningful.
# 3. TIME MATCHING. Frames were chosen as the three DEEPEST simulated
#    depressions, so the two columns were not the same instants and the beam had
#    travelled different distances. Frames are now matched in time.
#
# Frame times come from the file numbering: the camera runs at 50 kfps (20 us)
# and every 5th frame is stored. Solving t = (N - N0) * 20 us against the
# timestamps burned into the images gives N0 = 98.634, which reproduces all of
# them to under a microsecond. The simulated laser is on for 853 us, so only
# frames inside that window can be compared: files 110/125/140 -> 227/527/827 us.
# The original figure's 1.53 ms frame lay 0.7 ms past laser-off in the model.
import re

XRAY_TILT_DEG = 6.50          # measured, see above
FRAME0 = 98.634               # frame index of laser-on, from the burned-in times
FRAME_DT = 20e-6              # 50 kfps

def frame_time_s(path):
    m = re.search(r"G1200(\d+)\.tif$", str(path))
    return None if not m else (int(m.group(1)) - FRAME0) * FRAME_DT

def level(img, deg=XRAY_TILT_DEG):
    # +deg, not -deg. Array rows increase downward, which flips the sense of the
    # rotation: -6.5 deg DOUBLED the tilt (the darkest-row trace across columns
    # 300-900 went from a 68 px drop to 141 px). Verified by measuring that drop
    # after rotating: +6.5 deg leaves -5 to +1 px on every frame in the traverse,
    # while +7.5 deg overshoots to -10..-14 px.
    from scipy.ndimage import rotate as _rot
    return _rot(np.asarray(img, float), deg, reshape=False, order=1, mode="nearest")

frames = sorted((EXP / "xray").glob("*.tif"))
z_ti = cut_at_off(load_npz("bundle_gate", "t2_nist195"), "bundle_gate", "t2_nist195")
LASER_OFF_S = 853.2e-6

picks = [f for f in frames if (t := frame_time_s(f)) is not None and 0 < t < LASER_OFF_S]
picks = picks[::max(1, len(picks) // 3)][:3]

fig = plt.figure(figsize=(11, 6.4))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.15)

if picks:
    from PIL import Image
    for r, f in enumerate(picks):
        a = level(Image.open(f))
        axl = fig.add_subplot(gs[r, 0])
        axl.imshow(a, cmap="gray", aspect="equal")
        axl.set_xticks([]); axl.set_yticks([])
        axl.set_ylabel("%.0f us" % (frame_time_s(f) * 1e6), fontsize=9)
        if r == 0:
            axl.set_title("experiment - X-ray radiograph, levelled", fontsize=10)

if z_ti is not None and picks:
    F = z_ti["Fraction Of Fluid"]; zc = z_ti["z"]; x = z_ti["x"]
    j = F.shape[2] // 2
    ts = z_ti["time_s"]
    dser = depression_series(z_ti)
    for r, f in enumerate(picks):
        fr = int(np.argmin(np.abs(ts - frame_time_s(f))))
        axr = fig.add_subplot(gs[r, 1])
        # origin="lower" with an ascending z extent: low z at the bottom, so the
        # metal sits below the gas and the depression cuts downward.
        axr.imshow(F[fr, :, j, :].T, cmap="gray_r", origin="lower", aspect="equal",
                   extent=[x[0] * 1e6, x[-1] * 1e6, zc[0] * 1e6, zc[-1] * 1e6])
        axr.axhline(0, color=ACC, lw=0.8, ls=":")
        axr.set_xticks([]); axr.set_yticks([])
        if r == 0:
            axr.set_title("simulation - gas fraction, centreline slice", fontsize=10)
        axr.text(0.02, 0.08, "%.0f us, depth %.0f um" % (ts[fr] * 1e6, dser[fr]),
                 transform=axr.transAxes, fontsize=8, color=SIM,
                 bbox=dict(fc="white", ec="none", alpha=0.75, pad=1.5))
plt.show()
print("Rows are matched instants (%s us), both inside the 853 us laser-on window."
      % ", ".join("%.0f" % (frame_time_s(f) * 1e6) for f in picks))
print("Left: levelled radiographs (dark = vapour depression). Right: simulated gas fraction,")
print("low z at the bottom, so the depression cuts down from the dotted surface line.")


### 2b. The two superimposed, on one scale

Side-by-side panels share no axes, so they answer "does it look similar" but not
"is it the same size". This puts the simulated gas boundary on the radiograph with
both axes in microns and the aspect locked, so a length on one is a length on the
other.

Everything the overlay needs is measured rather than assumed, because none of it is
documented for these frames: the tilt (+6.50 deg, chosen by the residual it leaves,
not by the estimator that proposed it), the surface row (taken from the dark band
visible far ahead of the beam, so it is the substrate and not the track lying on
it), and the scale.

The scale was the hard part and was wrong twice. It is now fixed by an
over-determination: the beam position must be a straight line in time, since the
laser runs at a constant 0.7 m/s, and the slope of that line is fixed by physics
once a scale is chosen. That leaves one free parameter, the column where the laser
switched on, and only the correct scale makes all three frames agree on it
(109.12 +- 0.40 px at 2.01 um/px, against +-2.4 px at 1.96 and +-6.8 px at 2.19).
The scale is therefore good to about 1 %, and an unconstrained fit of the same
points returns 0.681 um/us against a true 0.700, an agreement that was not imposed.

**What it shows.** The trailing wake matches in every frame, with the rim at
z ~ +20 um in both. The cavity agrees at 627 us (60 um observed, 60 um simulated,
same position) and is minimal in both at 27 us. At 327 us the observed cavity is
115 um against a simulated 50 um, deeper and about 70 um further left. That is most
likely fluctuation rather than bias: the simulated depression oscillates 50-60 um
about a 55 +- 5 um mean, so a single experimental frame catching a deep excursion
is expected, and it is why every scored number in this notebook uses a steady-window
mean instead of a single frame. No conclusion should be drawn from one panel.


In [ ]:
# --------------------------------------------------- overlay, common scale --
# Side-by-side panels share no axes, so a reader cannot judge whether the
# depression is the same SIZE. This puts the simulated gas boundary onto the
# radiograph with both axes in microns and the aspect locked, so a length on one
# is a length on the other. Three constants, each measured rather than assumed:
#
#   TILT  +6.50 deg   the rotation that levels the frame. Verified by the
#                     residual, not by the estimator that produced it: the
#                     darkest-row trace across columns 300-900 drops 68 px raw,
#                     -5..+1 px at +6.5 deg, and -10..-14 px at +7.5 deg.
#   SURF  row 92      the substrate surface. Taken from the dark band that is
#                     present FAR AHEAD of the beam (deviation -14 from median at
#                     columns 700-800, against -122 inside the track), so it is
#                     the surface and not the melt track lying on it.
#   UMPX  1.96 um/px  from beam travel, not from a stated detector pitch (none is
#                     documented for these frames). The beam runs at 0.7 m/s, so
#                     the leading edge must advance 70 um per stored frame; the
#                     scale is 70 um divided by the measured advance in px.
#                     That advance depends on the detection threshold, so the
#                     threshold is chosen by FIT RESIDUAL rather than by taste:
#                       bg-10 -> 4.89 um/px, residual 112.6 px  (noise)
#                       bg-15 -> 1.96 um/px, residual   3.1 px  <- used
#                       bg-20 -> 2.19 um/px, residual   5.9 px
#                       bg-25 -> 2.32 um/px, residual  11.1 px
#                       bg-40 -> 3.06 um/px, residual  23.1 px
#                     The well-conditioned thresholds cluster at 1.96-2.19 um/px,
#                     so treat the scale as +-10 %. An earlier revision of this
#                     cell used 4.46 um/px, which came from the poorly conditioned
#                     end of that sweep and made the field 4.2 mm wide; 1.96 gives
#                     1.86 mm, which is the usual order for synchrotron
#                     radiography of a melt pool.
#
# Alignment: x = 0 is the beam. In the model that is the commanded position,
# start_point_m + V*t, NOT x[0] -- the domain starts 150 um before the beam does,
# and using x[0] shifted every frame by that much. In the radiograph it is the
# leading edge of the track within the surface band.
#
# READ THIS BEFORE DRAWING A CONCLUSION FROM THE OVERLAY. At the corrected scale
# the simulated cavity and the radiograph features are the same order and sit at
# the same x: at 327 us the simulated cavity reaches -55 um near x = -30, and the
# bright sub-surface region sits at x = -30..-10 reaching about -115 um. That is
# roughly a factor of two in depth, but it is NOT reported as a model error,
# because the bright region has not been identified. A 2.5-sigma threshold on it
# returns 923 / 227 / 477 / 254 / 156 um over the first five frames, and 923 um at
# t = 27 us is impossible, so it is not cleanly separable. Whether it is the
# vapour depression in projection, trapped porosity, or an imaging artifact is
# open. The overlay is a geometric comparison to look at, not a measured
# disagreement, and the +-10 % scale uncertainty applies to any length read off it.
TILT_DEG, SURF_ROW, UM_PER_PX = 6.50, 92, 2.01
X_ON_PX = 109.12       # pixel column where the laser switched on (fitted, see below)
TEXT_BOX = (slice(0, 30), slice(0, 200))   # burned-in timestamp; masked, not cropped
                                           # (cropping columns removed the track itself)
LASER_X0_M, LASER_V = -300e-6, 0.7        # laser.start_point_m, scan_speed_m_per_s

def _beam_px(t_s):
    """Beam column at time t. A LINE, not a per-frame detection.

    The leading-edge detector reported 144, 198, 225 px at 27/327/627 us -- steps
    of 54 then 27 px, i.e. a decelerating beam, which is wrong: the beam runs at a
    constant 0.7 m/s. Using it made the misalignment grow with time (+50, -50 and
    -200 um at those three frames). The beam position is therefore modelled as a
    line whose SLOPE is fixed by physics (V / scale) leaving one free parameter,
    the column where the laser switched on.

    That single parameter is over-determined by the three frames, which is what
    pins the scale: solving x_on at each frame and taking the spread gives
    106.21 +- 2.39 px at 1.96 um/px, 109.12 +- 0.40 px at 2.01, 111.34 +- 1.52 at
    2.05 and 118.48 +- 6.83 at 2.19. The minimum at 2.01 um/px is sharp, and the
    residual there is 0.8 um, so the scale is fixed to about 1 %. As a check the
    unconstrained fit of those same points gives 0.681 um/us against the true
    0.700, a 2.7 % agreement that was not imposed.
    """
    return X_ON_PX + (LASER_V * t_s * 1e6) / UM_PER_PX

if z_ti is not None and picks:
    from PIL import Image
    F = z_ti["Fraction Of Fluid"]; xc = z_ti["x"]; zc = z_ti["z"]; ts = z_ti["time_s"]
    j = F.shape[2] // 2
    fig, axes = plt.subplots(len(picks), 1, figsize=(9, 2.9 * len(picks)))
    for ax, f in zip(np.atleast_1d(axes), picks):
        t = frame_time_s(f)
        raw = np.asarray(Image.open(f), float).copy()
        raw[TEXT_BOX] = np.median(raw)         # blank BEFORE rotating: the text moves with it
        a = level(raw, TILT_DEG)
        x0 = _beam_px(t)
        ext = [(0 - x0) * UM_PER_PX, (a.shape[1] - x0) * UM_PER_PX,
               (a.shape[0] - SURF_ROW) * -UM_PER_PX, (0 - SURF_ROW) * -UM_PER_PX]
        ax.imshow(a, cmap="gray", aspect="equal", extent=ext, origin="upper")
        k = int(np.argmin(np.abs(ts - t)))
        beam = LASER_X0_M + LASER_V * ts[k]
        ax.contour((xc - beam) * 1e6, zc * 1e6, 1.0 - F[k, :, j, :].T,
                   levels=[0.5], colors=[SIM], linewidths=1.6)
        ax.axhline(0, color="0.3", lw=0.7, ls=":")
        ax.axvline(0, color="0.3", lw=0.7, ls=":")
        ax.set_xlim(-500, 200); ax.set_ylim(-180, 90)
        ax.set_title("%.0f us   radiograph + simulated gas boundary (blue)" % (t * 1e6),
                     fontsize=10)
        ax.set_xlabel("x from beam (um)", fontsize=9); ax.set_ylabel("z (um)", fontsize=9)
    plt.tight_layout(); plt.show()
    print("Both axes microns, aspect equal, beam at x = 0, substrate surface at z = 0.")
    print("Scale %.2f um/px and tilt %.2f deg are measured (see the cell comment), not assumed."
          % (UM_PER_PX, TILT_DEG))
    print("The bright sub-surface regions are NOT identified; do not read a depth from them.")


### 2c. Depth history behind those frames

Both traces on this plot are simulation. The solid line is the per-frame depression
depth and the dashed line is the mean of that same series over its steady window;
there is no measurement on this axis. The point of showing it is to establish that
the depression settles quickly and then oscillates about a stable value, which is
what justifies scoring a steady-window mean rather than any single frame.

It also exposes a defect in the rendered input worth knowing about. The shaded span
is the beam **traverse**, not "laser on": the path is 597 um at 0.7 m/s, so the
traverse ends at 853 us, but the turn-off was never written into the solver input.
The tables are single-row (`powlbm(1,1)=195`, `utlbm(1,1)=0.7`) and FLOW-3D holds
the last value, so the beam keeps firing and keeps moving to the end of the run,
travelling a further ~150 um into the domain edge. The peak temperature rising
after the nominal laser-off (3.58 -> 4.05 kK) is the giveaway, since with no heat
source it would fall. That tail is hatched and is excluded from every scored
metric, because `melt_metrics.depression` scores through
`steady_window(laser_off_s=...)`.


In [ ]:
# --------------------------------------------- depression depth history --
# Both curves are SIMULATION. There is no experimental trace on this axis: the
# solid line is the per-frame simulated vapour-depression depth, and the dashed
# line is the mean of that same simulated series over its steady window. The
# band is the simulation's own scatter, not a measurement uncertainty.
#
# The shaded span is the beam TRAVERSE, which is not the same as "laser on".
# The path runs x = -300 -> +297 um and at 0.7 m/s the traverse ends at 853 us,
# which is the `laser_off_time_s` recorded in the manifest. That turn-off was
# never rendered into the solver input: prepin carries single-row tables,
# `powlbm(1,1)=195` and `utlbm(1,1)=0.7`, and FLOW-3D holds the last value, so
# the beam keeps firing AND keeps moving to the end of the run. Between 853 and
# 1071 us it travels a further ~150 um into the domain edge, which is what lifts
# the depth and the peak temperature (3.58 kK -> 4.05 kK) at the right of the
# plot. That tail is an artifact of the overrun, not physics of a cooling pool.
#
# No reported number is affected: `melt_metrics.depression` scores through
# `steady_window(..., laser_off_s=...)`, which excludes every frame past 853 us.
# The tail is drawn, and hatched, so it cannot be mistaken for a settled result.
if z_ti is not None:
    TRAVERSE_END_MS = 0.8532        # 597.24 um path / 0.7 m/s, = derived.laser_off_time_s
    t_ms = z_ti["time_s"] * 1e3
    ds = depression_series(z_ti)
    on = z_ti["time_s"] < TRAVERSE_END_MS * 1e-3
    steady = ds[on][len(ds[on]) // 3:]
    fig, ax = plt.subplots(figsize=(7.8, 3.4))
    ax.plot(t_ms, ds, color=SIM, lw=1.5, label="simulation, per frame")
    ax.axvspan(0, TRAVERSE_END_MS, color=ACC, alpha=0.10, label="beam traversing")
    ax.axhline(steady.mean(), color=SIM, ls="--", lw=1,
               label="simulation, steady mean %.0f +/- %.0f um" % (steady.mean(), steady.std()))
    ax.set_xlabel("time (ms)"); ax.set_ylabel("vapour-depression depth (um)")
    ax.set_title("Simulated depression depth history - Ti64, 195 W, 0.7 m/s")
    ax.legend(fontsize=8, loc="lower right", framealpha=0.92)
    ax.set_ylim(0, max(ds) * 1.35); plt.show()
    reach = t_ms[int(np.argmax(ds >= 0.9 * steady.mean()))]
    print("Both traces are simulation; the dashed line is the mean of the solid one.")
    print("Depression establishes within ~%.0f us and holds %.0f +/- %.0f um across the traverse."
          % (reach * 1e3, steady.mean(), steady.std()))
    print("The rendered input never stops the beam, so the run continues past %.0f us;"
          % (TRAVERSE_END_MS * 1e3))
    print("every plot in this notebook is truncated there rather than showing the overrun.")


### 2d. Where in the scan does the model actually agree?

The three overlay panels are the SAME shot at different times, and the absorptance
trace in section 1 is that same shot too, recorded simultaneously: the radiograph
file numbers are the `FrameNumber` column of the absorptance file, and NIST's own
timestamps agree with the frame-time calibration used here to 0.3 us.

That makes a sharper test possible than any single frame. Below, the depression
depth measured from each radiograph is plotted against the absorbed fraction at the
same instant and against the simulated depth. Two things follow.

**The cavity drives the absorptance.** Depth and absorbed fraction rise and fall
together: deepest cavity at the absorptance peak, both decaying to a plateau. This
is the multiple-reflection picture in one experiment, and it is why the absorptance
transient in section 1 is a cavity transient.

**The model reproduces the settled keyhole, not the drilling transient.** Before
about 400 us the measured cavity is roughly twice the simulated one; after it, the
two agree within about 12 %, and at 627 and 827 us they agree to 1 % and 3 %. So
the middle overlay panel is not a representative failure -- it is a frame inside the
initial drilling excursion, where the real keyhole over-deepens before translation
establishes its steady shape. The scored metrics in this notebook average over the
steady window, which lies entirely in the regime where the model agrees; the
transient is a stated limit, not a hidden one.


In [ ]:
# ------------------------------- cavity depth vs absorptance, same shot --
# Depth is measured from the radiograph in a +-120 um window about the beam, below
# the surface, as the largest bright connected region (bright = less material in
# the path = cavity, see section 2). A minimum size of 25 px rejects speckle; the
# 27 us frame legitimately has no cavity, and its absorbed fraction (29.5 %) is the
# cold-surface value, so it is plotted as absent rather than as zero depth.
#
# The simulated absorbed fraction: the validated 44.3 % comes from the solver's
# energy audit in hd3out, but that file holds only 11 unique samples over the run,
# too coarse for a transient. The NPZ stores `Heat Absorption Rate` every frame, so
# the SHAPE comes from there and the SCALE from the audit (median ratio of audit
# power to summed field over 0.1-0.9 ms). Two caveats: it is NET absorptance, so it
# sits below a GROSS measurement by construction, and the audit-to-field shape
# correlation is 0.63, so read it as indicative. It reproduces the validated
# number, 45.0 % over 200-800 us against 44.3 %.
import pathlib as _pl
from scipy.ndimage import label as _label

_abs = pd.read_csv(EXP / "nist_ti64_scan_absorption.csv")
_t, _P, _A = (_abs["Time"].to_numpy() * 1e3, _abs["InputLaser"].to_numpy(),
              _abs["AbsoluteAbsorption"].to_numpy())

def _cavity_um(a, beam_px):
    sub = a[SURF_ROW + 2:, max(int(beam_px - 120 / UM_PER_PX), 0):int(beam_px + 40 / UM_PER_PX)]
    lab, n = _label(sub > np.median(a) + 2.0 * a.std())
    if not n:
        return np.nan
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    big = int(np.argmax(sizes))
    if sizes[big] < 25:
        return np.nan
    return (np.where((lab == big).any(axis=1))[0].max() + 2) * UM_PER_PX

def _sim_absorbed_pct(z, case_dir, power_w):
    import re
    H = z.get("Heat Absorption Rate")
    hits = glob.glob(str(case_dir / "hd3out.*"))
    if H is None or not hits:
        return None
    tot = H.sum(axis=(1, 2, 3)); tt = z["time_s"]
    txt = _pl.Path(hits[0]).read_text(errors="ignore")
    tm = [(m.start(), float(m.group(1)))
          for m in re.finditer(r"\btime\s*=\s*([0-9]\.[0-9Ee+-]+)", txt)]
    en = [(m.start(), float(m.group(1)))
          for m in re.finditer(r"total fluid #1 thermal energy\s*=\s*([0-9.Ee+-]+)", txt)]
    rows, ti = [], 0
    for pos, e in en:
        while ti + 1 < len(tm) and tm[ti + 1][0] < pos:
            ti += 1
        rows.append((tm[ti][1], e))
    a = np.array(sorted(set(map(tuple, rows))))
    if len(a) < 3:
        return None
    ta, E = a[:, 0], a[:, 1]
    keep = np.concatenate(([True], np.diff(ta) > 0)); ta, E = ta[keep], E[keep]
    dP = np.gradient(E, ta); ref = np.interp(ta, tt, tot)
    m = (ta > 1e-4) & (ta < 9e-4) & (ref > 0)
    if not m.any():
        return None
    return tt, 100.0 * tot * np.median(dP[m] / ref[m]) / power_w

if z_ti is not None and frames:
    from PIL import Image
    ser = mm.depression_series(z_ti); tsim = z_ti["time_s"]
    rows = []
    for f in frames:
        t = frame_time_s(f)
        if t is None or not (0 < t < tsim[-1]):
            continue
        raw = np.asarray(Image.open(f), float).copy(); raw[TEXT_BOX] = np.median(raw)
        a = level(raw, TILT_DEG)
        w = (_t >= t * 1e3 - 0.01) & (_t <= t * 1e3 + 0.01) & (_P > 50)
        rows.append(dict(t_us=t * 1e6,
                         absorbed=100 * _A[w].sum() / _P[w].sum() if w.any() else np.nan,
                         obs_um=_cavity_um(a, _beam_px(t)),
                         sim_um=ser[int(np.argmin(np.abs(tsim - t)))]))
    cdf = pd.DataFrame(rows)

    # (a) depression depth
    fig, axd = plt.subplots(figsize=(8.0, 3.6))
    axd.axvspan(0, 400, color=ACC, alpha=0.12)
    axd.plot(cdf.t_us, cdf.obs_um, "o-", color=EXPC, ms=7, label="measured (radiograph)")
    axd.plot(cdf.t_us, cdf.sim_um, "s--", color=SIM, ms=7, label="simulated")
    axd.set_xlabel("time (us)"); axd.set_ylabel("depression depth (um)")
    axd.set_title("(a) depression depth - Ti64 195 W, 0.7 m/s, one shot")
    axd.set_ylim(0, np.nanmax([cdf.obs_um.max(), cdf.sim_um.max()]) * 1.45)
    axd.text(200, axd.get_ylim()[1] * 0.92, "drilling transient", ha="center",
             fontsize=9, color="0.35")
    axd.legend(fontsize=8, loc="lower right", framealpha=0.92)
    plt.tight_layout(); plt.show()

    # (b) absorbed fraction
    fig, axa = plt.subplots(figsize=(8.0, 3.6))
    axa.axvspan(0, 400, color=ACC, alpha=0.12)
    axa.plot(cdf.t_us, cdf.absorbed, "^:", color=EXPC, ms=7,
             label="measured, gross (same shot)")
    _sim_abs = _sim_absorbed_pct(z_ti, RUNS / NIST_BATCH / "runs" / NIST_CASE, NIST_P)
    if _sim_abs is not None:
        _ta, _pa = _sim_abs
        _pk = np.convolve(_pa, np.ones(9) / 9, mode="same")
        _pk[:4] = _pa[:4]
        # Drop the last four samples rather than restoring the raw values there:
        # the moving average has no right-hand support, and the run's final frames
        # sit against the traverse end where the beam is closing on the boundary.
        axa.plot(_ta[:-4] * 1e6, _pk[:-4], "-", color=SIM, lw=1.5, alpha=0.85,
                 label="simulated, net (audit-scaled)")
    axa.set_xlabel("time (us)"); axa.set_ylabel("absorbed fraction (%)")
    axa.set_title("(b) absorbed fraction - same shot")
    axa.set_ylim(20, 88)
    axa.legend(fontsize=8, loc="lower left", framealpha=0.92)
    plt.tight_layout(); plt.show()

    cdf["ratio"] = cdf.obs_um / cdf.sim_um
    print(cdf.round(2).to_string(index=False))
    e = cdf[cdf.t_us < 400]; l = cdf[cdf.t_us >= 400]
    print()
    print("drilling transient (< 400 us): absorbed %.1f %%, measured/simulated %.2f"
          % (e.absorbed.mean(), e.ratio.mean(skipna=True)))
    print("settled            (>=400 us): absorbed %.1f %%, measured/simulated %.2f"
          % (l.absorbed.mean(), l.ratio.mean(skipna=True)))
    if _sim_abs is not None:
        _m = (_ta > 5e-5) & (_ta < 3e-4); _w = (_ta > 2e-4) & (_ta < 8e-4)
        print("simulated absorbed (net): %.1f %% cold surface, peaks %.1f %% near %.0f us, "
              "settles %.1f %%."
              % (_pk[np.argmin(np.abs(_ta - 25e-6))], _pk[_m].max(),
                 _ta[_m][np.argmax(_pk[_m])] * 1e6, _pk[_w].mean()))
    print("Depth and absorptance move together, so the section-1 absorptance transient")
    print("is a cavity transient. The model matches the settled keyhole and under-predicts")
    print("the initial drilling excursion by about a factor of two.")


### 2e. Every frame, and the same thing as an animation

The three panels in 2b are a sample. This is the whole sequence the comparison
rests on, so nothing is being chosen favourably: every stored radiograph from
laser-on to the end of the simulated run, levelled, beam-aligned and carrying the
simulated gas boundary on the same micron scale.

The sequence stops where the simulation does. Frames beyond it exist (the
experiment's laser runs to 1.96 ms and the radiographs continue to 2.43 ms) but
there is nothing to overlay on them, and the run's own last frames sit against the
domain boundary, so they are excluded rather than shown without a counterpart.

Read across the frames and the pattern in 2d is visible directly: through the
drilling transient the bright cavity is clearly deeper than the blue contour, and
from about 400 us the two track each other.


In [ ]:
# ------------------------------------------ every frame, plus an animation --
# Same geometry as 2b, applied to the whole sequence. `_panel` is shared by the
# contact sheet and the animation so the two cannot drift apart.
from matplotlib import animation
from IPython.display import HTML

_seq = [f for f in frames
        if frame_time_s(f) is not None and 0 < frame_time_s(f) <= z_ti["time_s"][-1]] \
       if (z_ti is not None and frames) else []

def _panel(ax, f, fontsize=7):
    from PIL import Image
    t = frame_time_s(f)
    raw = np.asarray(Image.open(f), float).copy(); raw[TEXT_BOX] = np.median(raw)
    a = level(raw, TILT_DEG)
    bx = _beam_px(t)
    ext = [(0 - bx) * UM_PER_PX, (a.shape[1] - bx) * UM_PER_PX,
           (a.shape[0] - SURF_ROW) * -UM_PER_PX, (0 - SURF_ROW) * -UM_PER_PX]
    ax.imshow(a, cmap="gray", aspect="equal", extent=ext, origin="upper")
    F = z_ti["Fraction Of Fluid"]; xc = z_ti["x"]; zc = z_ti["z"]; ts = z_ti["time_s"]
    k = int(np.argmin(np.abs(ts - t)))
    beam = LASER_X0_M + LASER_V * ts[k]
    ax.contour((xc - beam) * 1e6, zc * 1e6, 1.0 - F[k, :, F.shape[2] // 2, :].T,
               levels=[0.5], colors=[SIM], linewidths=1.2)
    ax.axhline(0, color="0.35", lw=0.5, ls=":"); ax.axvline(0, color="0.35", lw=0.5, ls=":")
    ax.set_xlim(-600, 250); ax.set_ylim(-200, 100)
    ax.set_xticks([]); ax.set_yticks([])
    return t

if _seq:
    ncol = 3; nrow = int(np.ceil(len(_seq) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 1.5 * nrow))
    for ax, f in zip(np.atleast_1d(axes).ravel(), _seq):
        t = _panel(ax, f)
        ax.set_title("%.0f us" % (t * 1e6), fontsize=8, pad=2)
    for ax in np.atleast_1d(axes).ravel()[len(_seq):]:
        ax.axis("off")
    fig.suptitle("Every frame from laser-on to the end of the simulated run; "
                 "blue = simulated gas boundary", fontsize=10)
    plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()
    print("%d frames, %.0f to %.0f us." % (len(_seq), frame_time_s(_seq[0]) * 1e6,
                                           frame_time_s(_seq[-1]) * 1e6))


In [ ]:
# The same frames as an animation, embedded so the notebook stays viewable without
# re-running it, and written to disk as an MP4 for use outside the notebook.
#
# ffmpeg: the SYSTEM binary is a snap and is confined -- it can encode but cannot
# write outside its allowed paths, which produced both "unable to open display"
# and a "Permission denied" on the output. `imageio_ffmpeg` ships an unconfined
# build inside the conda env, so matplotlib is pointed at that. With it, H.264
# works and the notebook stays small; the JS-HTML fallback embeds one PNG per
# frame and took this notebook from 1.5 to 4.4 MB for these nine frames.
if _seq:
    import matplotlib as _mpl
    try:
        import imageio_ffmpeg
        _mpl.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        pass

    figa, axa2 = plt.subplots(figsize=(7.4, 2.8), dpi=110)

    def _step(i):
        axa2.clear()
        t = _panel(axa2, _seq[i])
        axa2.set_title("t = %.0f us" % (t * 1e6), fontsize=10)

    _anim = animation.FuncAnimation(figa, _step, frames=len(_seq), interval=500)
    plt.close(figa)

    # MP4 beside the notebook, regenerated on every run so it cannot go stale.
    _mp4 = pathlib.Path.cwd() / "figs" / "keyhole_overlay.mp4"
    _mp4.parent.mkdir(exist_ok=True)
    try:
        _anim.save(str(_mp4), writer=animation.FFMpegWriter(
            fps=2, bitrate=2400,
            metadata={"title": "Keyhole overlay: NIST radiograph vs FLOW-3D",
                      "comment": "t2_nist195; scale 2.01 um/px, tilt +6.50 deg"}))
        print("wrote %s (%.2f MB)" % (_mp4, _mp4.stat().st_size / 1e6))
    except Exception as exc:                       # no usable ffmpeg
        print("MP4 export skipped: %s: %s" % (type(exc).__name__, exc))

    try:
        display(HTML(_anim.to_html5_video(embed_limit=20)))
    except Exception:                              # no ffmpeg: embedded PNGs
        display(HTML(_anim.to_jshtml(default_mode="loop")))


### 2f. Which parameter is responsible? Three controlled arms

Sections 2b-2e establish the disagreement without explaining it: the model reaches
the right cavity depth once the scan is steady, and misses it badly while the
keyhole is still drilling. Section 2d shows the same split in the absorptance,
which relaxes to the measured value while never reaching the measured early peak.
This section asks which model input is responsible, by moving one at a time.

The prompt was a suggestion that the deficit comes from too weak a temperature
dependence of surface tension. Most of that was already answered: sigma(T) here is
table-driven rather than a constant, its liquid slope is the Mills
-2.6e-4 N/m/K, and two earlier ablations found it inert. But every one of those
ablations scored the **steady fusion zone**, and there is one place the objection
survives that scoring. The Ti64 sigma table ends at 3123 K and holds constant above
it, while this case's vaporization temperature is 3520 K. Across the whole drilling
front, in exactly the window this section is about, dsigma/dT was zero by
construction and had never been tested.

Three arms, each differing from a control in exactly one **rendered solver input**.
The check is on the prepin rather than the config, because an earlier mesh
comparison in this project had to be withdrawn when its "pair" turned out to differ
in three physics switches as well as cell size:

| arm | prepin delta vs control |
|---|---|
| `t2n195_sigma4000` | 8 added `#sigma(1)` rows, extending past the clamp to 4000 K at the Mills liquid slope |
| `t2n195_tv3425` | `tv1` 3520 -> 3425 K |
| `t2n195_mesh5` | `size` 10 -> 5 um |

The control is the baseline re-rendered through the same pipeline. It exists
because the published run solved at 4 threads while the machine now caps at 10.
Threads do not change the physics, but leaving an uncontrolled difference inside a
matched pair is precisely how that earlier comparison went wrong.

The `tv1` arm is a **bound, not a refit**. `tv1` was calibrated against steady-state
depth at a different anchor, so fitting it to this transient would be circular.
Running it at the edge of its own stated uncertainty asks a different and
answerable question: how much of this gap could the vaporization calibration
account for while staying inside its anchor?

In [ ]:
# ------------------------ transient attribution: three arms and one control --
# Scored on the observable section 2c plots. Two windows: the drilling transient
# (120-400 us, where the disagreement lives) and a steady window running from
# 600 us to laser-off. The steady window STOPS at laser-off on purpose -- past it
# the cavity is collapsing, and including that tail drags the "steady" number down
# by several microns and would flatter every arm equally.
#
# Read the MEANS, not single frames. Depth is quantised to whole cells, so the
# 10 um arms resolve only 8-9 distinct depths across the entire run against 22 at
# 5 um; a per-frame change below one cell is invisible, while a shift in the mean
# is not. The `levels` column below is that resolution floor made visible.
ARM_BATCH = "t2_nist195_transient_20260806"
ARM_CASES = [("control (10 um)", "t2n195_ctrl"),
             ("sigma to 4000 K", "t2n195_sigma4000"),
             ("tv1 = 3425 K",    "t2n195_tv3425"),
             ("5 um cells",      "t2n195_mesh5")]

_arm_rows, arm_z = [], {}
for _label, _case in ARM_CASES:
    _z = load_npz(ARM_BATCH, _case)
    if _z is None:
        print("not exported yet:", _case)
        continue
    arm_z[_label] = _z
    _s = depression_series(_z)
    _t = np.asarray(_z["time_s"]).ravel()
    _tr = (_t >= 120e-6) & (_t <= 400e-6)
    _st = (_t > 600e-6) & (_t <= LASER_OFF_S)
    _arm_rows.append(dict(arm=_label, cell_um=cell_um(_z),
                          levels=int(len(np.unique(_s[_s > 0]))),
                          transient_um=float(_s[_tr].mean()),
                          steady_um=float(_s[_st].mean())))

arms_df = pd.DataFrame(_arm_rows)
if len(arms_df):
    _base = arms_df.iloc[0]
    arms_df["d_transient"] = arms_df.transient_um - _base.transient_um
    arms_df["d_steady"] = arms_df.steady_um - _base.steady_um
    # Lever > 1 means the arm moves the transient MORE than the steady state,
    # which is the signature a transient-specific cause has to show.
    arms_df["lever"] = np.where(arms_df.d_steady.abs() > 0.2,
                                arms_df.d_transient / arms_df.d_steady, np.nan)
    print(arms_df.round(2).to_string(index=False))

    if "cdf" in dir() and len(cdf):
        _m = cdf[(cdf.t_us >= 120) & (cdf.t_us <= 400)].obs_um
        _meas = float(np.nanmean(_m))
        print("\nMeasured transient mean over the same window: %.0f um (%d radiographs)."
              % (_meas, _m.notna().sum()))
        _gap = _meas - _base.transient_um
        print("Gap the control has to close: %.0f um." % _gap)
        for _, r in arms_df.iloc[1:].iterrows():
            print("  %-16s closes %+5.1f um = %+5.1f %% of it"
                  % (r.arm, r.d_transient, 100.0 * r.d_transient / _gap))


In [ ]:
# ------------------- all four arms on the same radiographs, in the transient --
# The table says one arm moves the cavity and two do not. This is where that
# either shows up to the eye or does not, using the SAME calibrated registration
# as 2b (tilt, surface row, scale and laser-on column are all reused from that
# cell, so this figure and the overlay above cannot drift apart).
#
# All four boundaries go on ONE frame rather than four panels side by side,
# because the question is which arm is deepest at the same instant, and that is
# hard to judge across separate axes.
if arm_z and frames:
    _picks = [f for f in frames
              if (tt := frame_time_s(f)) is not None and 0 < tt <= 400e-6]
    _picks = _picks[::max(1, len(_picks) // 4)][:4]
    _style = {"control (10 um)": ("#0072B2", "-"), "sigma to 4000 K": ("#E69F00", "--"),
              "tv1 = 3425 K": ("#009E73", "-."), "5 um cells": ("#D55E00", "-")}

    # Pull the mid-plane slice ONCE per arm. An NpzFile decompresses the whole
    # array on every key access, and the 5 um arm's Fraction Of Fluid is ~1 GB,
    # so reading it inside the frame loop would decompress a gigabyte 16 times.
    _mid = {}
    for _label, _z in arm_z.items():
        _F = _z["Fraction Of Fluid"]
        _mid[_label] = (np.asarray(_F[:, :, _F.shape[2] // 2, :]),
                        _z["x"], _z["z"], _z["time_s"])

    fig, axes = plt.subplots(len(_picks), 1, figsize=(9.5, 2.75 * len(_picks)))
    for ax, f in zip(np.atleast_1d(axes), _picks):
        t = frame_time_s(f)
        raw = np.asarray(Image.open(f), float).copy()
        raw[TEXT_BOX] = np.median(raw)
        a = level(raw, TILT_DEG)
        x0 = _beam_px(t)
        ext = [(0 - x0) * UM_PER_PX, (a.shape[1] - x0) * UM_PER_PX,
               (a.shape[0] - SURF_ROW) * -UM_PER_PX, (0 - SURF_ROW) * -UM_PER_PX]
        ax.imshow(a, cmap="gray", aspect="equal", extent=ext, origin="upper")
        for _label, (Fm, xc, zc, ts) in _mid.items():
            k = int(np.argmin(np.abs(ts - t)))
            beam = LASER_X0_M + LASER_V * ts[k]
            c, ls = _style[_label]
            ax.contour((xc - beam) * 1e6, zc * 1e6, 1.0 - Fm[k].T,
                       levels=[0.5], colors=[c], linewidths=1.5, linestyles=[ls])
        ax.axhline(0, color="0.3", lw=0.7, ls=":")
        ax.axvline(0, color="0.3", lw=0.7, ls=":")
        ax.set_xlim(-500, 200); ax.set_ylim(-180, 90)
        ax.set_title("%.0f us" % (t * 1e6), fontsize=10, loc="left")
        ax.set_xlabel("x from beam (um)", fontsize=9)
        ax.set_ylabel("z (um)", fontsize=9)
    np.atleast_1d(axes)[0].legend(
        handles=[plt.Line2D([], [], color=_style[l][0], ls=_style[l][1], lw=1.6, label=l)
                 for l in arm_z],
        loc="upper left", fontsize=8, framealpha=0.85, ncol=2)
    plt.tight_layout(); plt.show()
    print("Same registration as 2b. Beam at x = 0, substrate surface at z = 0.")
    print("The bright sub-surface region is NOT identified (see 2b): use this figure")
    print("for the RELATIVE ordering of the arms, and take absolute depths from 2c.")


**What the arms say.**

The surface-tension hypothesis is refuted, and refuted in the right regime. Carrying
sigma(T) past its clamp all the way to 4000 K moves the transient by a tenth of a
micron. That is a stronger statement than the earlier ablations supported, because
those scored steady fusion geometry while this one tests the drilling front above
the clamp, which is where the objection lived. The physics behind the null is
unsurprising in hindsight: trimming ~15 % off the capillary resistance is a mild
lever on cavity depth next to a recoil pressure that swings orders of magnitude
across the same temperature interval.

`tv1` does move the cavity, but as a near-uniform offset. It shifts the transient and
the steady state by almost the same amount, so it cannot explain a deficit that is
confined to the transient. It is worth noting that it *improves* the steady
agreement while leaving the transient far short, which is the opposite of what a
transient-specific cause would look like.

Cell size is the largest single mover and the only one that is transient-selective.
That was not the expected answer here. A controlled 10 um / 5 um pair elsewhere in
this project moved **fusion** depth by only four points, which is why the standing
verdict has been that the depth deficit is physics rather than resolution. That
verdict was earned on fusion geometry and does not carry to the transient vapour
cavity, where the same halving does considerably more. Resolution sensitivity is
observable-dependent, and this is a clean demonstration of it.

Two caveats belong with that number. The metric floor-rounds depth to whole cells,
so part of the improvement is the finer grid simply being able to *register* a depth
the coarse grid rounds away; the differential rounding bias is about half a cell.
And 5 um pushes the steady value to the top of the measured band, so some of the
agreement the 10 um model shows at steady state may be cancelling errors rather than
accuracy.

**The residual is the result.** No arm closes even a third of the transient gap, and
the three are not established as additive. Taken at face value they leave more than
half of it unattributed, which points away from all three of these parameters and
toward physics absent from the model during the drilling phase: no plume dynamics,
no Knudsen layer, a single-species vapour. That is now a bounded statement backed by
measurement rather than an unexplained gap, and it is the honest limit to quote.

These are sensitivity arms. Every published figure in this notebook stays on the
calibrated recipe, so the arms are a bound on it and not a replacement for it.

### 2g. Is the deficit general, or an artifact of the scanned case?

Section 2f leaves the transient gap bounded but mostly unexplained: surface tension is
a null, `tv1` is a near-uniform offset, cell size is the largest single mover at 31 %,
and over half of it is unaccounted for. Every one of those numbers comes from a single
shot -- the 195 W scan at 700 mm/s. A deficit measured once could be a property of the
recipe or a property of that geometry, and nothing so far separates the two.

The same NIST record holds a second configuration that was never used: a **stationary
2.0 ms pulse** on the same Ti-6Al-4V, recorded in the same session on the same
beamline. It shares the scanned case's beam optics exactly -- 122.5 um 1/e^2 at the
sample, so r0 = 61.25 um -- and its cold-surface absorptance is the same 30.5 %. It
differs in motion and in power (101.9 W against 195 W). The pair therefore isolates
**motion**, not optics, which is what makes it a test rather than another data point.

The condition is read from the source, not assumed (`validation/CITATIONS.md`, VERIFIED).
Power is the plateau median of the `InputLaser` trace; the README's first-few-hundred-ns
spike is a documented artifact and a naive maximum reads 311 W. Duration is the 2008.4 us
contiguous above-half-plateau run, which is also the energy-equivalent top-hat to within
6 us, so 2.008 ms conserves the delivered 204.1 mJ. The raw filenames say `t2077um`; that
is the **commanded** duration, and the delivered energy says 2.008 ms.

Two traps had to be cleared before anything could be compared.

**The radiograph scale does not come for free.** The scan's 2.01 um/px was recovered from
beam travel, and a stationary pulse has no travel. Neither the README nor the captioned
images carry a pixel size or a scale bar, and the only dimension in `Sample Diagram.pdf`
is the 300 um sample thickness, which lies along the X-ray path and is invisible in the
image plane. What settles it is that the record ships both representations of the *scan*:
comparing them shows the distributed "processed" images are a mirrored crop of the raw at
**identical sampling**, not a rebin -- the cavity travels -7.26 px/frame in the raw against
+7.94 px/frame in the processed, a ratio of 0.91 where a 2x rebin would give 0.5. So
2.01 um/px transfers to the spot frames, which are distributed raw only.

**The measurement dates an event, not just a depth.** Two independent channels agree on
when the keyhole forms: the cavity first exceeds 20 um at **867 us**, and the absorbed
fraction sits flat at 0.331 until 810 us then half-rises at **890 us**. One frame apart.
Before that the radiographs show only a dark lens straddling the surface -- displaced melt
piled *above* the original surface, adding material to the X-ray path -- and no sub-surface
cavity at all.

In [ ]:
# ------------------------- stationary spot: the measured target and the model --
# The radiograph reader lives in spot_radiographs.py beside this notebook rather
# than inline, because the spot needs its own geometry (640x600 raw ROI, -6.16 deg
# surface tilt, nothing moving) and the cavity has to be found by connectivity to
# the surface. A per-column bright-pixel scan was tried first and is not adequate:
# the vapour plume, the detector's banding and single hot pixels in the bulk are
# all bright too, and it read 0 um at 1007 us on a cavity that grows monotonically.
import spot_radiographs as sr

SPOT_DIR = EXP / "spot"
SPOT_CASE = "spot_nist102_2ms"
SPOT_POWER_W, SPOT_PULSE_US = 101.9, sr.PULSE_US
# Beam radius, checked against each case_manifest rather than assumed: all
# three spot cases carry beam_radius_m = 6.125e-05. Stating it beside a scale
# bar is what lets a reader SEE that the cavity bore is narrower than the
# beam, which is the bore argument made visible instead of asserted.
R0_UM = 61.25

z_spot = load_npz("validation", SPOT_CASE)
# The matched mesh pair, on the reduced 500x500x400 um box. The 10 um arm of the
# pair is the control for the domain change, so panels must draw IT rather than
# the full-domain run alongside the 5 um arm.
z_spot_10 = load_npz("validation", "spot_nist102_2ms_sm_10um")
z_spot_5 = load_npz("validation", "spot_nist102_2ms_sm_5um")
meas = np.array([(t, d) for _, t, d in sr.depth_series(SPOT_DIR)])
meas = meas[(meas[:, 0] >= 0) & (meas[:, 0] <= SPOT_PULSE_US)]
a_meas = sr.absorptance(SPOT_DIR)

if z_spot is None:
    print("spot case not exported yet")
else:
    t_us = z_spot["time_s"] * 1e6
    sim_d = mm.depression_series(z_spot)
    print("cell %.1f um | %s" % (cell_um(z_spot), mm.domain_validity(z_spot)))
    print("peak T %.0f K (tv1 = 3520 K) -- the model DOES reach vaporization"
          % z_spot["Temperature"].max())

    rows = []
    for tt in (200, 400, 600, 800, 900, 1000, 1200, 1500, 1800, 2000):
        i = int(np.argmin(np.abs(meas[:, 0] - tt)))
        j = int(np.argmin(np.abs(t_us - tt)))
        m, s = meas[i, 1], sim_d[j]
        rows.append({"t (us)": tt, "measured (um)": round(m, 1),
                     "simulated (um)": round(float(s), 1),
                     "meas/sim": round(m / s, 2) if s > 5 else np.nan})
    display(pd.DataFrame(rows).set_index("t (us)"))
    print("measured peak %.0f um at %.0f us | simulated peak %.0f um at %.0f us"
          % (meas[:, 1].max(), meas[np.argmax(meas[:, 1]), 0],
             sim_d.max(), t_us[np.argmax(sim_d)]))

In [ ]:
# --------------------------------- net (model) against gross (measured) coupling --
# The solver writes its energy audit every 220 us, so this is as fine as the model
# side gets without a rerun. NET vs GROSS is not a like-for-like comparison in
# absolute level -- net counts only energy still in the metal -- so what carries the
# argument is the SHAPE: whether the coupling doubles when the keyhole forms.
def _sim_absorptance(run_dir, power_w, pulse_s):
    hits = sorted(glob.glob(str(pathlib.Path(run_dir) / "hd3out.*")))
    if not hits:
        return np.empty((0, 2))
    txt = pathlib.Path(hits[0]).read_text(errors="ignore")
    times = [(m.start(), float(m.group(1)))
             for m in re.finditer(r"\btime\s*=\s*([0-9]\.[0-9Ee+-]+)", txt)]
    en = [(m.start(), float(m.group(1)))
          for m in re.finditer(r"total fluid #1 thermal energy\s*=\s*([0-9.Ee+-]+)", txt)]
    rows, ti = [], 0
    for pos, e in en:
        while ti + 1 < len(times) and times[ti + 1][0] < pos:
            ti += 1
        if not rows or times[ti][1] != rows[-1][0]:
            rows.append((times[ti][1], e))
    out = []
    for (t0, e0), (t1, e1) in zip(rows, rows[1:]):
        # Drop the interval straddling laser-off: the laser is on for only part of
        # it, and it plots as a spurious collapse to zero.
        if t1 > pulse_s:
            break
        out.append(((t0 + t1) / 2 * 1e6, (e1 - e0) / (power_w * (t1 - t0))))
    return np.array(out)

SPOT_RUN = RUNS.parent / "validation" / SPOT_CASE
a_sim = _sim_absorptance(SPOT_RUN, SPOT_POWER_W, SPOT_PULSE_US * 1e-6)

if z_spot is not None and len(a_sim):
    fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.9))

    ax[0].plot(meas[:, 0], meas[:, 1], color=EXPC, lw=1.6, label="measured (X-ray)")
    ax[0].plot(t_us, sim_d, color=SIM, lw=1.6, label="simulated (depression)")
    ax[0].axvline(867, color=ACC, ls="--", lw=1, label="measured keyhole onset")
    ax[0].axhline(10, color="0.6", ls=":", lw=0.9)
    ax[0].text(60, 12, "one cell (10 um)", fontsize=7.5, color="0.4")
    ax[0].set_xlim(0, SPOT_PULSE_US); ax[0].set_xlabel("time (us)")
    ax[0].set_ylabel("cavity depth below original surface (um)")
    ax[0].set_title("(a) depth through the pulse"); ax[0].legend(loc="upper left")

    ax[1].plot(a_meas[:, 0], a_meas[:, 3], color=EXPC, lw=1.4, label="measured, gross")
    ax[1].step(a_sim[:, 0], a_sim[:, 1], where="mid", color=SIM, lw=1.6,
               label="simulated, net (220 us audit)")
    ax[1].axvline(890, color=ACC, ls="--", lw=1)
    ax[1].set_xlim(0, SPOT_PULSE_US); ax[1].set_ylim(0, 0.8)
    ax[1].set_xlabel("time (us)"); ax[1].set_ylabel("absorbed fraction")
    ax[1].set_title("(b) absorbed fraction"); ax[1].legend(loc="upper left")

    gd = np.interp(meas[:, 0], t_us, sim_d)
    ok = (gd > 5) & (meas[:, 0] > 200)
    ax[2].plot(meas[ok, 0], meas[ok, 1] / gd[ok], color="k", lw=1.6)
    ax[2].axhline(1, color="0.5", lw=0.9)
    ax[2].axhline(2.12, color=EXPC, ls=":", lw=1.1)
    ax[2].text(1150, 2.25, "scanned case's worst (2.12x)", fontsize=8, color=EXPC)
    ax[2].axvline(867, color=ACC, ls="--", lw=1)
    ax[2].set_xlim(0, SPOT_PULSE_US); ax[2].set_xlabel("time (us)")
    ax[2].set_ylabel("measured / simulated depth")
    ax[2].set_title("(d) depth ratio: it diverges, it does not converge")
    fig.suptitle("NIST mds2-2525 stationary spot vs %s (10 um, calibrated recipe)"
                 % SPOT_CASE, fontsize=10)
    plt.show()

    # Average only over audit intervals lying FULLY before onset. The interval
    # that straddles it already contains the keyhole (0.534 on the 5 um run) and
    # averaging it into a "before onset" number inflated it from 0.356 to 0.401
    # in the first version of this section.
    SPOT_ONSET_SIM_US = 675.0
    lo = a_sim[a_sim[:, 0] < SPOT_ONSET_SIM_US]
    hi = a_sim[a_sim[:, 0] > 880]
    print("net simulated coupling: %.3f before onset -> %.3f after"
          % (lo[:, 1].mean(), hi[:, 1].mean()))
    print("gross measured coupling: %.3f before onset -> %.3f after"
          % (a_meas[a_meas[:, 0] < 880, 3].mean(), a_meas[a_meas[:, 0] > 880, 3].mean()))

In [ ]:
# ------------------- (c) the simulated gas boundary drawn on the radiographs --
# Registration is measured on the pre-pulse mean and printed, so the figure can be
# checked rather than trusted: 2.01 um/px, tilt -6.16 deg (fit sd 0.34 px over all
# 640 columns), surface at row 92.7 after levelling, beam column from the mean x of
# the late-pulse cavity. Levelling is by -6.16 deg, not +6.16: rotating the other
# way doubles the tilt to -12.3 deg, which is the same sign trap section 2b hit.
from scipy.ndimage import rotate as _rot

# ------------------------------------------------- visual scale bar (shared) --
# Copied verbatim from the manuscript repo's notebooks/calibration_figures.ipynb
# so both repos draw an identical bar. Do not reimplement: the choices matter.
#   - DATA coordinates, so the bar is length_um wide BY CONSTRUCTION and stays
#     correct under resize, aspect change or crop. An axes-fraction bar becomes a
#     lie the moment the limits move, which is the usual failure.
#   - Call it AFTER set_xlim/set_ylim: it reads the limits to place itself.
#   - solid_capstyle="butt": the default round cap adds half a linewidth at each
#     end, a real error on a short bar.
#   - zorder=5 to sit over an imshow; white reads on both the raw and the
#     flat-fielded radiograph (pass colour="k" on a light background).
def scale_bar(ax, length_um=100, pad=0.055, colour='w'):
    """Visual length reference, drawn after the limits are fixed."""
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    xe = x1 - pad * (x1 - x0); xs = xe - length_um
    y = y0 + pad * (y1 - y0)
    ax.plot([xs, xe], [y, y], color=colour, lw=3.0, solid_capstyle='butt',
            zorder=5)
    ax.text((xs + xe) / 2, y + 0.035 * (y1 - y0), f'{length_um:g} ' + r'$\mu$m',
            color=colour, ha='center', va='bottom', fontsize=11, zorder=5)


def verify_scale_bar(fig, ax, length_um=100, pad=0.055):
    """Assert the drawn bar really spans length_um in data units.

    Cheap and worth running once per new axes rather than trusting the helper:
    it catches an axes-fraction regression or an aspect surprise immediately.
    """
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    xe = x1 - pad * (x1 - x0); xs = xe - length_um
    y = y0 + pad * (y1 - y0)
    # RATIO of two measurements sharing one transform, NOT an absolute px/um.
    # transData is stale before a canvas draw -- measured 1.0197 px/um pre-draw
    # against 0.9333 after constrained_layout settles the axes, 9 % off -- so an
    # absolute check either lies or forces a draw. The ratio cancels the
    # staleness exactly, so this is correct pre-draw and costs no draw, which is
    # what makes it safe to call inside a per-frame video loop.
    tr = ax.transData
    span_px = (tr.transform((xe, y)) - tr.transform((xs, y)))[0]
    unit_px = (tr.transform((x0 + 1.0, y)) - tr.transform((x0, y)))[0]
    implied = span_px / unit_px
    assert abs(implied - length_um) < 1e-6, (implied, length_um)
    return implied, unit_px


_TILT = -6.164
_files, _ref, (_a, _b, _sd), _dpx = sr.geometry(SPOT_DIR)
_lref = _rot(_ref, _TILT, reshape=False, order=1, mode="nearest")
_, _surf_row, _ = sr.surface_fit(_lref)
_late = _rot(sr._img(_files, 175) / np.maximum(_ref, 1), _TILT, reshape=False,
             order=1, mode="nearest")
_H, _W = _late.shape
_yy, _xx = np.mgrid[0:_H, 0:_W]
_m = (_late > sr.THRESH) & (_yy > _surf_row + 6)
_beam_col = float(_xx[_m].mean())
print("registration: %.2f um/px, tilt %+.2f deg, surface row %.1f, beam col %.1f"
      % (sr.UM_PER_PX, _TILT, _surf_row, _beam_col))

# Both meshes on every panel. This function predates the mesh pair and drew only
# the full-domain 10 um run, which made the video show a lone dimple and no
# indication that the 5 um arm exists -- the whole point of the section.
# Okabe-Ito, matching the manuscript's mesh colours: 10 um blue, 5 um orange.
C10, C5 = "#0072B2", "#E69F00"
_MESH_ARMS = [(z_spot_10, C10, 2.6, "10 um cells"),
              (z_spot_5, C5, 1.5, "5 um cells")]

def _spot_panel(ax, n, half_um=250.0, deep_um=320.0, legend=False, bar=True):
    img = _rot(sr._img(_files, n) / np.maximum(_ref, 1), _TILT, reshape=False,
               order=1, mode="nearest")
    ext = [(0 - _beam_col) * sr.UM_PER_PX, (_W - _beam_col) * sr.UM_PER_PX,
           (_H - _surf_row) * -sr.UM_PER_PX, (0 - _surf_row) * -sr.UM_PER_PX]
    ax.imshow(img, cmap="gray", origin="upper", extent=ext, aspect="equal",
              vmin=0.85, vmax=1.25)
    t = sr.frame_time_us(n)
    for _z, _col, _lw, _lab in _MESH_ARMS:
        if _z is None:
            continue
        k = int(np.argmin(np.abs(_z["time_s"] * 1e6 - t)))
        F = _z["Fraction Of Fluid"][k]
        # The 0.5 iso-contour of fluid fraction on the centreline slice IS the
        # gas/metal interface. Drawn, not filled, so the radiograph stays visible.
        ax.contour(_z["x"] * 1e6, _z["z"] * 1e6, F[:, F.shape[1] // 2, :].T,
                   levels=[0.5], colors=[_col], linewidths=_lw)
        if legend:
            # Proxy handle: matplotlib 3.8+ dropped ContourSet.collections, so a
            # contour can no longer label itself.
            ax.plot([], [], color=_col, lw=_lw, label=_lab)
    ax.axhline(0, color=EXPC, lw=0.7, ls=":")
    if legend:
        ax.plot([], [], color=EXPC, lw=0.7, ls=":", label="original surface")
        ax.legend(loc="lower left", fontsize=8, framealpha=0.85, handlelength=1.6)
    ax.set_xlim(-half_um, half_um); ax.set_ylim(-deep_um, 90)
    ax.set_xticks([]); ax.set_yticks([])
    # After the limits, never before. The time stamp lives in the TITLE here, so
    # the default bottom-right placement cannot collide with it.
    if bar:
        scale_bar(ax, length_um=100)
    return t

_picks = [110, 145, 165, 183]
figc, axc = plt.subplots(1, len(_picks), figsize=(3.0 * len(_picks), 3.4))
for _i, (_ax, _n) in enumerate(zip(axc, _picks)):
    _ax.set_title("%.0f us" % _spot_panel(_ax, _n, legend=(_i == 0)), fontsize=10)
axc[0].set_ylabel("depth (um)")
figc.suptitle(r"Radiograph with both simulated gas boundaries, beam $r_0$ = %.2f $\mu$m "
              "(1/e$^2$). The 10 um dimple never leaves the surface; the 5 um cavity "
              "tracks the measurement." % R0_UM, fontsize=10)
# Verify the bar spans what it claims, on the real axes, once.
_imp, _ppu = verify_scale_bar(figc, axc[0], length_um=100)
print("scale bar: implied %.6f um (target 100), %.4f px/um; 100 um = %.2f detector px "
      "at %.2f um/px" % (_imp, _ppu, 100 / sr.UM_PER_PX, sr.UM_PER_PX))
plt.show()

In [ ]:
# The same thing as a video across the pulse and its collapse, embedded so the
# notebook stays viewable without re-running, and written to figs/ for use
# outside it.
#
# SPOT_VIDEO_FPS is the one knob. 4 fps is slow enough to read the keyhole
# forming frame by frame; the frames are 2 x 19.987 us apart, so 4 fps plays
# ~100x slower than real time. Raise it for a smoother sweep, lower it to dwell
# on the onset.
SPOT_VIDEO_FPS = 6
SPOT_VIDEO_STRIDE = 2          # every 2nd camera frame -> ~40 us per video frame

import matplotlib as _mpl
try:
    import imageio_ffmpeg
    _mpl.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
except Exception:
    pass

_seq = list(range(82, 210, SPOT_VIDEO_STRIDE))
figv, axv = plt.subplots(figsize=(5.2, 4.0), dpi=110)

def _spot_step(i):
    axv.clear()
    # axv.clear() above removes the previous frame's artists, so drawing the bar
    # per frame does not stack them. length_um is FIXED and the limits never
    # change between frames, so the bar means the same thing throughout.
    t = _spot_panel(axv, _seq[i], legend=True, bar=True)
    axv.set_title(r"NIST spot 101.9 W, $r_0$ = 61.25 $\mu$m, 10 vs 5 um  |  t = %.0f us%s"
                  % (t, "" if t <= SPOT_PULSE_US else "  (laser off)"), fontsize=10)

_spot_anim = animation.FuncAnimation(
    figv, _spot_step, frames=len(_seq), interval=1000.0 / SPOT_VIDEO_FPS)
plt.close(figv)

_mp4 = pathlib.Path.cwd() / "figs" / "spot_overlay.mp4"
_mp4.parent.mkdir(exist_ok=True)
try:
    _spot_anim.save(str(_mp4), writer=animation.FFMpegWriter(
        fps=SPOT_VIDEO_FPS, bitrate=3000,
        metadata={"title": "NIST mds2-2525 spot: radiograph vs FLOW-3D",
                  "comment": "spot_nist102_2ms; 2.01 um/px, levelled -6.16 deg"}))
    print("wrote %s (%.2f MB, %d frames at %d fps)"
          % (_mp4, _mp4.stat().st_size / 1e6, len(_seq), SPOT_VIDEO_FPS))
except Exception as exc:
    print("MP4 export skipped: %s: %s" % (type(exc).__name__, exc))

try:
    display(HTML(_spot_anim.to_html5_video(embed_limit=30)))
except Exception:
    display(HTML(_spot_anim.to_jshtml(default_mode="loop")))

**What the spot says.**

At 10 um the model never makes the keyhole transition: the measured cavity reaches
161 um while the simulated depression never exceeds 30 um -- three cells -- across
the whole 2.008 ms pulse. At 5 um, on the same domain and differing in nothing but
cell size, it drills 195 um in 39 cells. Halving the cell does not improve the
answer by 31 %; it changes which regime the model is in.

The coupling says the same thing independently, and it is not quantised. Averaged
over the audit intervals lying fully before onset, both meshes land within 5 % of
the measured **absolute** level -- 0.348 (10 um) and 0.356 (5 um) against a measured
0.331 -- with nothing fitted to it. After onset the measurement doubles to 0.62, the
5 um run follows to 0.55, and the 10 um run stays at 0.29 and drifts down. So the
optics and the flat-surface coupling are right, and what is missing at 10 um is the
specific gain of a keyhole: multiple reflections trapped in a deep cavity. The
measured bore is 24-42 um, two to four cells at 10 um, and a ray cannot bounce down
a channel that narrow.

**Depth agreement, not overshoot.** The 5 um peak of 195 um against a measured 161 um
is *inside the measurement's own spread*: sweeping the flat-field threshold from 1.06
to 1.20 moves the measured peak over 157-192 um, and the late-pulse keyhole genuinely
oscillates by +-31 um. The honest statement is that at 5 um the model stops falling
short; whether it overshoots is below the precision of this comparison.

**Timing is the real discrepancy.** The simulated onset is 675 us against a measured
867 us -- about ten frames early. That one survives the threshold sweep (867 us at
every threshold but the loosest) and is corroborated at 890 us by the absorptance
channel, which is a physically independent measurement. A timing error outside the
data's own scatter is a finding in a way a depth difference inside it is not.

**Scope: this is the extreme case, not the general one.** Resolution is decisive
*here*, in a 2 ms stationary dwell, and only partial in every scanned case measured:
the t2_nist195 pair moves the transient depression 45 -> 72 um, the 90 W pair of
section 2f closes about four points of a 53 % deficit, and the Galiegue fusion ladder
is a continuous gradient (-59 % at 5 cells deep to -9 % at 22) rather than a
threshold. Long dwell at one point is what lets the feedback run away; a moving beam
never dwells long enough for the cavity to reach the depth where its own geometry
takes over. The training corpus is scanned, so the correct statement is that 10 um
under-resolves the keyhole transient everywhere and progressively worse as the pool
gets shallower -- not that scanned training data cannot represent keyholing.

---
## 3. Melt-pool geometry - IN718, NIST AM Bench AMB2022-03

Cross-sectioned single tracks on bare IN718, no powder. Reference values are means
over the individual measurements in the NIST spreadsheet (n = 6 per variant, error
bars = 1 s.d.), computed by `fetch_experimental_data.py` rather than read off a
figure. The simulated fusion zone is the ever-melted region at mid-track - what
metallography reveals after solidification.

All six cases run the **IN718 recipe**: `in718_full` properties, per-material
Fresnel, mass-flow recoil, pv0 = 1 atm - no plume, no sigma-table extension.

**Calibration disclosure.** The two component drops were decided by matched-pair
ablations scored against the **285 W base condition of this same table**: the
sigma extension only widened the pool (+1 cell, no depth effect) and the plume
imposed a near-uniform 20-30 um depth cut at every condition (the energy audits
in section 7 show the diverted fraction is deleted, not redistributed). The
285 W row below is therefore the *selection* condition; the other five rows
were solved after selection without further adjustment, but they are
neighbouring conditions of the same dataset, machine and material lot - a
transfer check, not independent validation.

**Out-of-sample blind test (pre-registered, reported verbatim).** The seventh
condition of the NIST table - 285 W, 0.96 m/s, 49 um spot, the dataset's one
deep-keyhole variant (targets 227.2/106.2 um, aspect 2.14) - was never used in
any calibration step and was solved blind with the frozen recipe:
**170/120 um -> depth -25.2 % (below the -20 % band), width +13.0 % (in
band)**. The plume arm is worse (140/140, depth -38.4 %). The recipe that is
essentially exact across the transition-regime table under-drills the keyhole
condition in both arms, so the IN718 validated scope is the
transition/conduction regime; deep-keyhole IN718 is out of scope, with the
saturation-curve/vapour-identity family as the identified suspect (section 7).
For context, the published distribution for blind keyhole-depth prediction on
NIST benchmarks spans 1.5-6x errors (Simonds et al., IMMI 2024); this blind
miss is 1.25x.

In [ ]:
CASES = [  # label, batch, case, P, V, key into the NIST target table
    ("245 W",        "bundle_gate", "rep3_in718_p245",   245.0, 0.96, "(245.0, 960.0, 67.0)"),
    ("285 W (base)", "bundle_gate", "p3a_in718_noplume", 285.0, 0.96, "(285.0, 960.0, 67.0)"),
    ("325 W",        "bundle_gate", "rep3_in718_p325",   325.0, 0.96, "(325.0, 960.0, 67.0)"),
    ("0.80 m/s",     "bundle_gate", "rep3_in718_v080",   285.0, 0.80, "(285.0, 800.0, 67.0)"),
    ("1.20 m/s",     "bundle_gate", "rep3_in718_v120",   285.0, 1.20, "(285.0, 1200.0, 67.0)"),
    ("82 um spot",   "bundle_gate", "rep3_in718_s082",   285.0, 0.96, "(285.0, 960.0, 82.0)"),
]
tfile = EXP / "amb718_targets.json"
targets = json.loads(tfile.read_text()) if tfile.exists() else {}

rows = []
for label, batch, case, P, V, key in CASES:
    z = cut_at_off(load_npz(batch, case), batch, case)
    if z is None or key not in targets:
        continue
    d_sim, w_sim = fusion_zone(z)
    t = targets[key]
    rows.append(dict(label=label, P=P, V=V, d_sim=d_sim, d_exp=t["depth_um"], d_sd=t["depth_sd"],
                     w_sim=w_sim, w_exp=t["width_um"], w_sd=t["width_sd"],
                     half_cell=cell_um(z) / 2))
df = pd.DataFrame(rows)
if len(df):
    df["d_err"] = 100 * (df.d_sim - df.d_exp) / df.d_exp
    df["w_err"] = 100 * (df.w_sim - df.w_exp) / df.w_exp
    df["d_ok"] = [band(r.d_sim, r.d_exp, r.d_sd) for r in df.itertuples()]
    df["w_ok"] = [band(r.w_sim, r.w_exp, r.w_sd) for r in df.itertuples()]
    # a leg within half a cell (5 um) of its band edge is indeterminate (section 0)
    def edge(sim, exp, sd, rel=0.20, half_cell=None):
        b = max(rel * exp, 2 * sd)
        return abs(abs(sim - exp) - b) < (half_cell or 5.0)
    df["d_edge"] = [edge(r.d_sim, r.d_exp, r.d_sd, half_cell=r.half_cell) for r in df.itertuples()]
    df["w_edge"] = [edge(r.w_sim, r.w_exp, r.w_sd, half_cell=r.half_cell) for r in df.itertuples()]
    print(df[["label", "d_sim", "d_exp", "d_sd", "d_err", "d_ok", "d_edge",
              "w_sim", "w_exp", "w_sd", "w_err", "w_ok", "w_edge"]].round(1).to_string(index=False))
    ind = int(df.d_edge.sum() + df.w_edge.sum())
    lo = int(df.d_ok.sum() + df.w_ok.sum()) - int((df.d_ok & df.d_edge).sum() + (df.w_ok & df.w_edge).sum())
    print("\nstrict count: %d-%d of %d legs in band (%d edge-indeterminate at the 10 um floor)"
          % (lo, lo + ind, 2 * len(df), ind))
else:
    print("IN718 final-recipe cases not available yet")

In [ ]:
if len(df):
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    hi = max(df[["d_sim", "d_exp", "w_sim", "w_exp"]].max()) * 1.15
    lim = [0, hi]
    ax[0].plot(lim, lim, color="0.4", lw=1, ls="--", label="1:1")
    ax[0].fill_between(lim, [v * 0.8 for v in lim], [v * 1.2 for v in lim],
                       color="0.6", alpha=0.15, label="+/-20 %")
    ax[0].errorbar(df.d_exp, df.d_sim, xerr=df.d_sd, fmt="o", color=SIM, ms=7,
                   capsize=3, label="depth")
    ax[0].errorbar(df.w_exp, df.w_sim, xerr=df.w_sd, fmt="s", color=ACC, ms=6,
                   capsize=3, label="width")
    ax[0].set_xlim(lim); ax[0].set_ylim(lim)
    ax[0].set_xlabel("measured (um)"); ax[0].set_ylabel("simulated (um)")
    ax[0].set_title("(a) parity - IN718 melt pool"); ax[0].legend()

    p = df[df.label.isin(["245 W", "285 W (base)", "325 W"])].sort_values("P")
    ax[1].errorbar(p.P, p.d_exp, yerr=p.d_sd, fmt="o-", color=EXPC, capsize=3, label="experiment")
    ax[1].plot(p.P, p.d_sim, "s--", color=SIM, label="simulation")
    ax[1].set_xlabel("laser power (W)"); ax[1].set_ylabel("melt depth (um)")
    ax[1].set_title("(b) depth vs power (0.96 m/s)"); ax[1].legend()

    v = df[df.label.isin(["0.80 m/s", "285 W (base)", "1.20 m/s"])].sort_values("V")
    ax[2].errorbar(v.V, v.d_exp, yerr=v.d_sd, fmt="o-", color=EXPC, capsize=3, label="experiment")
    ax[2].plot(v.V, v.d_sim, "s--", color=SIM, label="simulation")
    ax[2].set_xlabel("scan speed (m/s)"); ax[2].set_ylabel("melt depth (um)")
    ax[2].set_title("(c) depth vs speed (285 W)"); ax[2].legend()
    plt.show()

    if len(p) == 3:
        print("depth growth 245->325 W:   experiment %+.0f %%   simulation %+.0f %%"
              % (100 * (p.d_exp.iloc[-1] - p.d_exp.iloc[0]) / p.d_exp.iloc[0],
                 100 * (p.d_sim.iloc[-1] - p.d_sim.iloc[0]) / p.d_sim.iloc[0]))
    if len(v) == 3:
        print("depth change 0.8->1.2 m/s: experiment %+.0f %%   simulation %+.0f %%"
              % (100 * (v.d_exp.iloc[-1] - v.d_exp.iloc[0]) / v.d_exp.iloc[0],
                 100 * (v.d_sim.iloc[-1] - v.d_sim.iloc[0]) / v.d_sim.iloc[0]))

---
## 4. Keyhole depth - Ti-6Al-4V, Gan et al. 2021

Synchrotron X-ray measurements of vapour-depression depth at fixed speed and spot,
stepping the power through the conduction -> keyhole transition. These are the
quantitative counterpart to the radiographs in section 2, and they probe the
regime the recoil-pressure model governs most strongly.

The four simulations run the **final Ti64 recipe**: per-material Fresnel,
mass-flow recoil, pv0 = 1 atm, sigma to Tv, and a saturation anchor
tv1 = 3520 K - **no plume**. tv1 is the recipe's one calibrated scalar: the
material databases' 3123 K (with pure-Ti molar mass and latent heat) is the
most volatile consistent convention and over-drilled +48 % at 129 W; tv1 was
scanned on that single condition (3123/3315/3420/3520 K -> 126/107/99/93 um vs
85.3) and frozen inside its literature bracket (alloy convention 3315 K to pure Ti
3560 K). Honest precision: the scan noise maps to **+- ~95 K (1 sigma)** on
tv1, the unconstrained best fit (~3600 K) lies above the bracket and was
clipped to it, and +8.8 % residual remains at the calibration condition. What
makes 3520 K physically defensible is not ideal Raoult (whose effective anchor,
~3420 K, is the losing arm) but **measured Al activities in Ti melts**
(gamma_Al ~ 0.05 -> effective anchor 3513-3653 K). **Only 129 W is a
calibration point; 103, 156 and 206 W and all of section 5 are post-selection
checks of that single scalar.** 156 W was never used for parameter selection,
but its pass gated adoption under the pre-registered rule - it is a
post-selection check, not held-out confirmation.
Simulated points are steady-window means +- 1 sd; the depth oscillates
+-30-40 um frame to frame, so single-frame comparisons would sit inside the
noise.

An earlier model version achieved comparable anchor accuracy with a
fixed-fraction plume attenuation (aplume = 0.20) instead of the corrected
anchor; the review's energy audits showed that component acted as a
beam-power sink compensating the over-volatile curve, and it is retired from
every recipe (section 7).

In [ ]:
gfile = EXP / "gan_points.json"
gan = json.loads(gfile.read_text()) if gfile.exists() else []
# Bundle-physics solves of the Gan ladder (0.4 m/s, 95 um 1/e^2 spot = 47.5 um radius)
GAN_CASES = [("t2_103w", 103.198), ("t1_129w_tv3520_noplume", 129.360),
             ("t2_156w", 155.523), ("t2_206w", 206.395)]

krows = []
for case, P in GAN_CASES:
    z = cut_at_off(load_npz("bundle_gate", case), "bundle_gate", case)
    ref = [g for g in gan if g["group"] and "Ti64" in g["group"]
           and abs(g["P"] - P) < 0.5 and abs(g["V"] - 0.4) < 1e-6]
    if z is None or not ref:
        continue
    d_sim, w_sim, d_sd = depression(z)
    krows.append(dict(P=P, d_sim=d_sim, d_sd=d_sd, d_exp=ref[0]["depth_um"], w_sim=w_sim))
kdf = pd.DataFrame(krows)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
# Panel (a) is one comparison plus a backdrop, and the two must not look alike.
# The claim is the 0.4 m/s ladder: four measured depths and the four simulated
# depths at the same power, spot and speed. Everything else in Gan's Ti64 set is
# backdrop showing where those four sit in the published range.
#
# The backdrop is 48 points spanning SEVEN scan speeds (0.4 to 1.2 m/s) at one
# spot size (r0 = 47.5 um, verified: every Ti64 row carries that radius). Depth
# depends on power AND speed, so plotted against power alone those points are not
# a curve -- they are seven curves superimposed, which is why the spread looks
# like scatter. Drawn in one neutral grey with a single legend entry saying so.
# Previously each speed got its own marker in the SAME colour as the comparison,
# and `[:4]` labelled only four of the seven, leaving three series unlabelled.
if len(gan):
    ti = pd.DataFrame([g for g in gan if g["group"] and "Ti64" in g["group"]])
    ax[0].plot(ti.P, ti.depth_um, "o", ms=3.5, alpha=0.35, color="0.6",
               markeredgewidth=0,
               label="Gan 2021, all Ti64 points (V = %.1f-%.1f m/s, context only)"
                     % (ti.V.min(), ti.V.max()))
if len(kdf):
    ax[0].plot(kdf.P, kdf.d_exp, "o-", color=EXPC, ms=8, lw=1.5, label="measured, V = 0.4 m/s (the scored ladder)")
    ax[0].errorbar(kdf.P, kdf.d_sim, yerr=kdf.d_sd, fmt="s--", color=SIM, ms=8, lw=1.8,
                   capsize=3, label="simulation (steady window +- sd)")
ax[0].set_xlabel("laser power (W)"); ax[0].set_ylabel("keyhole depth (um)")
ax[0].set_title("(a) keyhole depth vs power - Ti64, 95 um spot, V = 0.4 m/s"); ax[0].legend(fontsize=8)

if len(kdf):
    lim = [0, max(kdf[["d_sim", "d_exp"]].max()) * 1.2]
    ax[1].plot(lim, lim, "--", color="0.4", lw=1, label="1:1")
    ax[1].fill_between(lim, [v * 0.8 for v in lim], [v * 1.2 for v in lim],
                       color="0.6", alpha=0.15, label="+/-20 %")
    ax[1].errorbar(kdf.d_exp, kdf.d_sim, yerr=kdf.d_sd, fmt="none", ecolor="0.5",
                   capsize=3, zorder=4)
    sc = ax[1].scatter(kdf.d_exp, kdf.d_sim, c=kdf.P, cmap="viridis", s=90, zorder=5)
    plt.colorbar(sc, ax=ax[1], label="power (W)")
    ax[1].set_xlim(lim); ax[1].set_ylim(lim)
    ax[1].set_xlabel("measured keyhole depth (um)"); ax[1].set_ylabel("simulated (um)")
    ax[1].set_title("(b) parity - keyhole depth"); ax[1].legend(fontsize=8)
plt.show()

if len(kdf):
    kdf["err"] = 100 * (kdf.d_sim - kdf.d_exp) / kdf.d_exp
    print(kdf[["P", "d_sim", "d_sd", "d_exp", "err"]].round(1).to_string(index=False))
    print()
    print("(a) the red line is the measured 0.4 m/s ladder and the blue line is the simulation")
    print("    at those same four conditions; grey is every other Gan Ti64 point, at speeds up")
    print("    to 1.2 m/s, shown only to place the ladder inside the published range. Grey is")
    print("    not a trend: depth varies with speed as well as power.")
    print("(b) the same four pairs against 1:1, coloured by power, with the +/-20 % band.")
else:
    print("keyhole cases still solving - rerun this cell when the bundle_gate batch finishes")

---
## 5. Conduction -> keyhole transition - stationary spot

Simonds et al. report, for a stationary 2 ms exposure on bare Ti64, a vapour
depression appearing near **96 W** and a true keyhole near **120 W**. This is the
sharpest available test of the transition because it isolates the onset from any
scanning effect. The simulations use the runner's stationary-spot mode
(`laser.dwell_time_s`) at 1 ms exposure.

Read the two ends of the ladder differently. At **90 W** the simulation staying
conduction-mode is consistent with the measured kinetics: Allen et al. (2020)
report keyhole incubation times diverging near threshold (~2.4 ms at 90 W), so
a 1 ms exposure should not keyhole there, and it does not. At **105 W** the
same argument runs the other way and is *not* satisfied: the simulation reaches
aspect ratio ~1.3 at half the experimental exposure, where the experiment shows
depression-only until ~120 W - the model keyholes **at least 15 W early**, and
the depth at 105 W is still rising when the laser switches off (103-128 um
depending on the scoring window). This is a mild REGRESSION vs the retired sink
arm (aspect 1.19, 92-117 um): suppressing evaporative cooling makes the pool
~730 K hotter, trading onset accuracy for ladder parsimony - disclosed, not
excused. The transition is graded rather than a step,
which is an improvement over earlier physics, but its position errs on the
over-drilling side and the kinetics argument must not be used to excuse it.

In [ ]:
# ------------------------------------------- stationary onset of drilling --
# "VD" = VAPOUR DEPRESSION: the gas-filled dimple the recoil pressure pushes into
# the melt surface. Two measured thresholds are marked, and they are different
# events. The first is the power at which a depression appears at all. The second
# is the power at which that depression becomes a KEYHOLE, conventionally when it
# is at least as deep as it is wide (aspect >= 1), which is when multiple internal
# reflections start trapping the beam. Panel (b) is that criterion.
#
# THE LASER DOES NOT TURN OFF IN THESE RUNS. The config asks for a 1 ms dwell and
# the manifest records laser_off_time_s = 1.0 ms, but the turn-off is never
# rendered: prepin carries a single-row table, powlbm(1,1)=90 with timlbm(1)=0,
# and FLOW-3D holds the last value, so the beam fires for the whole 1.15 ms run.
# A stationary beam that never stops keeps drilling, so the extra 150 us is not
# free. Scoring the same cases with the window closed at 1.0 ms moves the depth by
# about 2 um and the aspect by 0.01 (90 W: 23.0 -> 23.2 and 0.26 -> 0.26;
# 105 W: 120.7 -> 122.9 and 1.24 -> 1.25; 130 W: 197.3 -> 200.3 and 1.85 -> 1.86),
# so the conclusion below does not rest on the overrun. The window is closed at
# the intended dwell anyway, which is the honest choice.
#
# ERROR BARS ARE TEMPORAL SCATTER, NOT MESH. They are the standard deviation of
# the per-frame depth (and width) across the steady window, i.e. how much the
# cavity oscillates while it is nominally steady. Discretisation is a SEPARATE
# and unshown floor: these run at 10 um cells, so any length also carries about
# +-5 um of quantisation. At 90 W the depth is 23 um, i.e. only ~2 cells, and the
# 5.3 um scatter is about one cell -- that point is resolution-limited and should
# not be read as a precise depth.
#
# Panel (b) previously had no error bars because the metric returns a depth sd and
# no width sd. Both are computed here over the same steady window and propagated
# in quadrature, which matters: at 105 W the aspect is 1.26 +- 0.24, so the +-1 sd
# range is 1.02 to 1.50. The model is above the keyhole criterion there, but only
# just, and a bare marker at 1.24 hid that.
DWELL = [("t2_dwell90", 90.0), ("t2_dwell105", 105.0), ("t2_dwell130", 130.0)]
DWELL_OFF_S = 1.0e-3            # intended dwell; the rendered input never stops

drows = []
for case, P in DWELL:
    z = cut_at_off(load_npz("bundle_gate", case), "bundle_gate", case)
    if z is None:
        continue
    ser = mm.depression_series(z)
    win = mm.steady_window(len(ser), laser_off_s=DWELL_OFF_S, times_s=z["time_s"])
    ds, ws = [], []
    for k in np.arange(len(ser))[win]:
        g = mm.depression_frame(z, int(k))
        ds.append(g.depth_um); ws.append(g.width_um)
    ds, ws = np.array(ds), np.array(ws)
    keep = ws > 0
    asp = ds[keep] / ws[keep]
    drows.append(dict(P=P, depth_um=ds.mean(), depth_sd=ds.std(),
                      width_um=ws.mean(), width_sd=ws.std(),
                      aspect=asp.mean(), aspect_sd=asp.std(),
                      cell_um=float((z["z"][1] - z["z"][0]) * 1e6),
                      tmax_K=float(z["Temperature"].max())))
ddf = pd.DataFrame(drows)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a in ax:
    a.axvspan(96, 120, color=ACC, alpha=0.15)
ax[0].axvline(96, color=EXPC, ls=":", lw=1.5,
              label="measured vapour-depression onset ~96 W")
ax[0].axvline(120, color=EXPC, ls="--", lw=1.5, label="measured keyhole onset ~120 W")
if len(ddf):
    ax[0].errorbar(ddf.P, ddf.depth_um, yerr=ddf.depth_sd, fmt="o-", color=SIM, ms=8,
                   capsize=3, label="simulated depth (bars = scatter in time)")
ax[0].set_xlabel("laser power (W)"); ax[0].set_ylabel("vapour-depression depth (um)")
ax[0].set_title("(a) onset of drilling - stationary, 1 ms dwell"); ax[0].legend(fontsize=8)

ax[1].axhline(1.0, color="0.4", ls="--", lw=1, label="keyhole criterion (depth = width)")
if len(ddf):
    ax[1].errorbar(ddf.P, ddf.aspect, yerr=ddf.aspect_sd, fmt="s-", color=SIM, ms=8,
                   capsize=3, label="simulated aspect (bars = scatter in time)")
ax[1].set_xlabel("laser power (W)"); ax[1].set_ylabel("aspect ratio (depth / width)")
ax[1].set_title("(b) keyhole criterion"); ax[1].legend(fontsize=8)
plt.show()

if len(ddf):
    print(ddf[["P", "depth_um", "depth_sd", "width_um", "width_sd",
               "aspect", "aspect_sd", "cell_um", "tmax_K"]].round(2).to_string(index=False))
    print()
    print("Bars are the cavity's oscillation in time, NOT a mesh estimate; at 10 um cells")
    print("every length also carries ~+-5 um of quantisation, and the 90 W depth is ~2 cells.")
    print("The laser is never switched off in the rendered input, so these 1 ms dwells fire")
    print("for the full 1.15 ms run; the window is closed at 1.0 ms here, which moves depth")
    print("by ~2 um and aspect by ~0.01.")
if len(ddf) == 3:
    print()
    print("These are 1 ms exposures; Simonds measured at 2 ms. Dedicated 2 ms runs show BOTH")
    print("recipe arms keyholing at 105 W (aspect 1.35-1.59, stable across every scoring window)")
    print("where the experiment reports depression-only until ~120 W: the model enters the")
    print("keyhole regime ~15 W early, robustly and independently of the recipe. No onset POWER")
    print("or depth is claimed - the cavity is still growing at laser-off even at 2 ms.")


---
## 6. Melt-pool geometry - CoCr, De Araujo Soares et al. 2026

A second alloy, and the point of it is that nothing about the recipe is refitted
to reach it. CoCr takes its own vaporisation anchor (`tv1 = 3200 K`) and its own
Fresnel coefficient from the material file; the rest of the physics is the recipe
validated on Ti64. Cross-sectioned single tracks on bare plate, widths and depths
being means over eight sections, so the reference carries real scatter rather than
a single reading.

Widths are the paper's primary observable and depths its secondary one, which is
worth keeping in mind when reading the two rows: the width agreement is the
stronger claim.


In [ ]:
# CoCr: fusion depth and width against the sectioned tracks.
COCR = [("cocr_p120_v04", 121.0, 0.40), ("cocr_p200_v095", 200.0, 0.95),
        ("cocr_p290_v095", 290.0, 0.95)]
_pts = sa.normalize_points(sa.load_anchor(EXP / "appsci_cocr_targets.json"))

rows = []
for case, P, V in COCR:
    z = cut_at_off(load_npz("bundle_gate", case), "bundle_gate", case)
    hit = [p for p in _pts if abs(p["power_w"] - P) < 1.0 and abs(p["speed_m_s"] - V) < 1e-6]
    if z is None or not hit:
        continue
    d, w = fusion_zone(z)
    rows.append(dict(P=P, V=V, d_sim=d, d_exp=hit[0]["depth_um"],
                     w_sim=w, w_exp=hit[0]["width_um"]))
cocr_df = pd.DataFrame(rows)

if len(cocr_df):
    cocr_df["d_err"] = 100 * (cocr_df.d_sim - cocr_df.d_exp) / cocr_df.d_exp
    cocr_df["w_err"] = 100 * (cocr_df.w_sim - cocr_df.w_exp) / cocr_df.w_exp
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    for k, (a, lab) in enumerate(zip(ax, ("depth", "width"))):
        sim = cocr_df.d_sim if k == 0 else cocr_df.w_sim
        exp = cocr_df.d_exp if k == 0 else cocr_df.w_exp
        lim = [0, max(sim.max(), exp.max()) * 1.25]
        a.plot(lim, lim, "--", color="0.45", lw=1, label="1:1")
        a.fill_between(lim, [v * (1 - mm.DEFAULT_BAND) for v in lim],
                       [v * (1 + mm.DEFAULT_BAND) for v in lim],
                       color="0.6", alpha=0.15, label=f"+/-{mm.DEFAULT_BAND*100:.0f} %")
        a.scatter(exp, sim, c=cocr_df.P, cmap="viridis", s=80, zorder=5)
        a.set_xlim(lim); a.set_ylim(lim)
        a.set_xlabel(f"measured {lab} (um)"); a.set_ylabel(f"simulated {lab} (um)")
        a.set_title(f"CoCr fusion {lab}")
        a.legend(fontsize=8, loc="upper left", framealpha=0.9)
    plt.tight_layout(); plt.show()
    print(cocr_df.round(1).to_string(index=False))
    print()
    print("depth : %d/%d in band, mean signed %+.0f %%"
          % (sum(abs(cocr_df.d_err) <= mm.DEFAULT_BAND * 100), len(cocr_df), cocr_df.d_err.mean()))
    print("width : %d/%d in band, mean signed %+.0f %%"
          % (sum(abs(cocr_df.w_err) <= mm.DEFAULT_BAND * 100), len(cocr_df), cocr_df.w_err.mean()))
    print("Every point is inside the band, but the depth error is one-signed across all")
    print("three conditions, so it is a systematic offset and not scatter. Reporting the")
    print("bias alongside the pass count is required by validation/ACCEPTANCE.md.")
else:
    print("CoCr cases not available")


---
## 7. Stationary spot welds - SS304L, Hawk & Simonds 2022

The third alloy, and the only stationary arm scored against a published section.
These are 10 ms dwells at 118-456 W, so the laser never translates and `V_scan = 0`
is the measurement rather than a missing value. That is exactly what made them
hard to ingest: the guard that stops a dropped scan speed becoming a silent zero
refused them until it learned to accept a declared-stationary case.

**The series name matters.** Hawk reports four SS304L material variants whose
depths differ by up to a factor of two at the same power, so a comparison that
pools them is meaningless. Only `wrought_304L_AR_GR` is scored here.


In [ ]:
# SS304L stationary spot welds, one named material variant.
SS_SERIES = "wrought_304L_AR_GR"
SS = [("s304t2_p118", 118.0), ("s304t2_p255", 255.0),
      ("s304t2_p355", 355.0), ("s304t2_p456", 456.0)]
_sp = sa.normalize_points(sa.load_anchor(EXP / "hawk304l_targets.json"), series=SS_SERIES)

rows = []
for case, P in SS:
    z = cut_at_off(load_npz("bundle_gate", case), "bundle_gate", case)
    hit = [p for p in _sp if abs(p["power_w"] - P) < 1.0]
    if z is None or not hit:
        continue
    d, w = fusion_zone(z)
    # A pool that reaches the bottom of its box is not a measurement of depth.
    # 456 W melts 775 um in a domain whose floor is at -787.5 um, so
    # `domain_validity` refuses it; the enlarged-domain rerun
    # (s304t3_p456_bigdom, floor at -1587.5 um) returns 825 um and IS valid, so
    # the depth is real, it just cannot be scored from this case.
    rows.append(dict(P=P, d_sim=d, d_exp=hit[0]["depth_um"],
                     w_sim=w, w_exp=hit[0]["width_um"],
                     domain_ok=bool(mm.domain_validity(z).ok)))
ss_df = pd.DataFrame(rows)

if len(ss_df):
    ss_df["d_err"] = 100 * (ss_df.d_sim - ss_df.d_exp) / ss_df.d_exp
    ss_df["w_err"] = 100 * (ss_df.w_sim - ss_df.w_exp) / ss_df.w_exp
    fig, ax = plt.subplots(figsize=(7.6, 3.8))
    ax.plot(ss_df.P, ss_df.d_exp, "o-", color=EXPC, ms=8, label="measured depth")
    _ok = ss_df.domain_ok
    ax.plot(ss_df.P, ss_df.d_sim, "--", color=SIM, lw=1.5, zorder=1)
    ax.plot(ss_df.P[_ok], ss_df.d_sim[_ok], "s", color=SIM, ms=8, label="simulated depth")
    if (~_ok).any():
        ax.plot(ss_df.P[~_ok], ss_df.d_sim[~_ok], "s", mfc="none", ms=11,
                color=SIM, mew=1.8, label="simulated, pool reaches the domain floor")
    ax.set_xlabel("laser power (W)"); ax.set_ylabel("fusion depth (um)")
    ax.set_title("SS304L, %s, 10 ms stationary dwell" % SS_SERIES.replace("_", " "))
    ax.legend(fontsize=9, loc="upper left", framealpha=0.9)
    plt.tight_layout(); plt.show()
    print(ss_df.round(1).to_string(index=False))
    print()
    _v = ss_df[ss_df.domain_ok]
    print("depth : %d/%d in band, mean signed %+.0f %%  (all four rungs)"
          % (sum(abs(ss_df.d_err) <= mm.DEFAULT_BAND * 100), len(ss_df), ss_df.d_err.mean()))
    print("        %d/%d in band, mean signed %+.0f %%  (excluding the domain-limited 456 W)"
          % (sum(abs(_v.d_err) <= mm.DEFAULT_BAND * 100), len(_v), _v.d_err.mean()))
    print("width : %d/%d in band, mean signed %+.0f %%"
          % (sum(abs(ss_df.w_err) <= mm.DEFAULT_BAND * 100), len(ss_df), ss_df.w_err.mean()))
    print("The depth error changes sign across the ladder rather than holding one way, so")
    print("this is a spread rather than a bias; the model tracks the trend but not every rung.")
    print("The 456 W pool reaches its domain floor, so that rung is shown open and reported")
    print("separately; the enlarged-domain rerun gives 825 um and is domain-valid.")
else:
    print("SS304L cases not available")


---
## 8. Summary

Generated from the figures above, so it cannot drift out of sync with them.

In [ ]:
summary = []
if np.isfinite(sim_frac) and np.isfinite(meas_qs):
    summary.append(dict(quantity="absorbed fraction (Ti64, 195 W)",
                        simulation="%.1f %%" % (sim_frac * 100),
                        experiment="%.1f %%" % (meas_qs * 100),
                        agreement="%+.1f pts" % ((sim_frac - meas_qs) * 100),
                        source="NIST mds2-2525"))
if len(df):
    summary.append(dict(quantity="IN718 melt depth (%d conditions)" % len(df),
                        simulation="-", experiment="-",
                        agreement="%.0f %% mean |err|, %d/%d in band"
                                  % (df.d_err.abs().mean(), df.d_ok.sum(), len(df)),
                        source="NIST mds2-2718"))
    summary.append(dict(quantity="IN718 melt width (%d conditions)" % len(df),
                        simulation="-", experiment="-",
                        agreement="%.0f %% mean |err|, %d/%d in band"
                                  % (df.w_err.abs().mean(), df.w_ok.sum(), len(df)),
                        source="NIST mds2-2718"))
if len(kdf):
    summary.append(dict(quantity="Ti64 keyhole depth (%d powers)" % len(kdf),
                        simulation="-", experiment="-",
                        agreement="%.0f %% mean |err|" % kdf.err.abs().mean(),
                        source="Gan 2021"))
if len(ddf):
    hit = ddf[ddf.aspect >= 1.0]
    summary.append(dict(quantity="keyhole onset (stationary)",
                        simulation=("%.0f W" % hit.P.min()) if len(hit) else "not reached <=130 W",
                        experiment="~120 W", agreement="-", source="Simonds 2021"))
sdf = pd.DataFrame(summary)
print(sdf.to_string(index=False) if len(sdf) else "no completed cases yet")

---
## 9. The retired plume component, and the model's validity scope

**How the recipes lost their one ad-hoc component.** An earlier model version
carried a fixed-fraction plume attenuation (aplume = 0.20, rfplume = 8x spot)
for Ti64. The adversarial review established from the solver's own energy
ledger that it acted as a **beam-power sink**: matched-pair audits across four
material/condition pairs show the melt receives ~aplume less energy (deficits
9-16 points of beam power) and ACONDENS returns none of it (bit-identical
ablation at 90 W; measured null at 129 W, audits 56.1 vs 55.6 %). Its scope
was a single point - Ti64, 47.5 um spot, keyhole regime - and it degraded
every other anchor: a near-uniform 20-30 um depth cut on all IN718 conditions
(the 82 um case's apparent insensitivity was the cut cancelling the
defocused-Gaussian over-drill), failures in both arms at 61.25 um, and worse
SS316/IN718-keyhole under-drilling.

**What it was compensating - found and fixed.** The Ti64 material carried
tv1 = 3123 K with pure-Ti molar mass and latent heat: the most volatile
consistent convention, giving 3x the IN718/SS316 saturation pressure at
4000 K where an ideal-Raoult estimate puts Ti64 lowest. A single-condition tv1
scan at 129 W, no plume (3123/3315/3420/3520 K -> 126/107/99/93 um vs 85.3)
identified tv1 = 3520 K (+- ~95 K; the unconstrained best fit ~3600 K was
clipped to the bracket; measured Al activities in Ti melts, gamma_Al ~ 0.05,
put the physical effective anchor at 3513-3653 K) - and the anchor replay
passed **5/5 falsifiable legs** with no plume (103 W is floor-indeterminate;
105 W landed between the registered outcome bands, window-dependent
103-128 um): dwells 23/119/198 um (onset graded, domain-independent), ladder
-12 % (floor-indeterminate) / +2.0 / +3.2 %, and the out-of-scope 61 um
absorptance moved from 39.8 % (sink) and 58.9-60.8 % (old curve) to 44.3 %
against 48.5 % measured. **No recipe - Ti64, IN718, SS316, or the synthetic
sweep materials that inherit Ti64's vapour identity - carries a plume.** The
aplume/rfplume/acondens knobs remain implemented for future use.

**Calibration status of tv1 = 3520 +- ~95 K:** one calibrated scalar, fitted
on one condition (Gan 129 W - a point that has now been scored under nine
model configurations across the campaign), checked post-selection on six
further anchors. 156 W was never used for selection but its pass gated
adoption; any tv1 in ~[3450, 3560] would produce statistically equivalent
data at these anchors. On the truly post-selection keyhole points (156/206 W)
the calibrated arm reads 2.6 % mean |err| vs the retired sink arm's 5.1 %.

**Onset.** The transition grades (23/119/198 um at 90/105/130 W). The 90 W
leg is consistent with measured incubation kinetics (1 ms < the ~2.4 ms
incubation); at 105 W the model still reaches aspect ~1 at half the measured
exposure where the experiment shows depression-only until ~120 W - it errs on
the over-drilling side near threshold, stated as such (section 5). Eliminated
as onset-sharpness causes by matched controls: accommodation coefficient,
recoil coefficient form, void vapour pressure, Marangoni extension, ray
count, beam-radius convention, mesh 10 -> 5 um.

**Remaining under-drilling family (IN718 keyhole, SS316) - cause OPEN.** The
blind IN718 49 um keyhole fails at -25.2 % (best arm) and SS316 at -37 %.
A saturation-curve correction was the natural suspect, but it fails
quantitatively: ideal-Raoult multi-species estimates provide x1.7-2.1 in
p_sat where the deficits require x5.5-12 at the ladder's measured
14.8 %/e-fold depth sensitivity - a Raoult-corrected SS316 curve would
recover only ~+5-6 % of the -37 % (registered prediction). Candidates beyond
the curve: the absorption chain (ereflct tables end ~1900 K; hot-liquid
resistivity rise un-modeled), property-table clamps, mesh.

**Thermal-envelope limit.** The calibrated arm runs keyhole walls at
4459-4700 K - 460-700 K above the ~4000 K measured LPBF envelope (two-colour
pyrometry) - with every property table clamped far below the peaks (mu/rho/cv
end 2173 K, thc 2500 K, ereflct 1923 K, sigma 3123 K). The depth anchors
absorb these clamps silently; absolute temperatures from this model should
not be trusted, and the softer curve largely relabels wall temperature rather
than changing drilling pressure (p_sat at the hotter walls reaches the same
tens of atmospheres).

**Precedent.** All seven scanned-geometry submissions to the 2022 AM Bench
asynchronous challenge over-predicted keyhole depth by 1.5-6x and missed
keyhole-onset timing by up to ~6x in either direction (Simonds et al., IMMI
2024, doi:10.1007/s40192-023-00336-0); keyhole-depth errors of 50-200 % are
typical unless measured real-time absorptance is imposed (Zhang et al., Nat
Commun 2024); a FLOW-3D benchmark against the same Ti-6Al-4V dataset reports
36-59 % keyhole-length errors with recoil and evaporation terms as the
dominant uncertainty (Samaei et al., 2024). Against that distribution, this
model's in-scope keyhole errors (a few %) and its blind out-of-sample keyhole
miss (25 %) sit at the favorable end - claimed as such, not as uniform
validity.

### 10. Model justification ledger and limits

Every component that adds complexity over the vendor keyhole profile, with the
evidence that keeps it. "Ablated" means a matched-control pair isolates the
component. One agreement band (+-20 % or 2 sd, section 0) is used for every
count in this notebook.

| component | applies to | what it replaces | evidence | status |
|---|---|---|---|---|
| per-material Fresnel (`ereflct` from resistivity/absorptivity) | all | one GUI-era scalar for all materials | root cause of the historical 0/300-keyhole sweep; absorptance direction vs NIST | **keep** |
| mass-flow recoil (`recoil_model=2`) | all | analytic A exp(B(1-Tv/T)) overpressure | dwell ladder passes and the onset grades; analytic form was -5 % only | **keep** |
| `pv0 = 101325 Pa` | all | GUI relic 50650 Pa | physical correctness (ambient partial pressure); isolated effect -5 % | **keep** |
| per-material `tvexp = R/(M*Lv)` | all | legacy hardcoded 5.989e-5 (17x-soft recoil for Ti64) | vendor's own recipe; falsified legacy curve | **keep** (default material carries M since 2026-07-29, user-approved) |
| Ti64 saturation anchor `tv1 = 3520 +- ~95 K` | Ti64 (`materials/ti64_v3_calibrated`); synthetics sample their own vapour identities and are NOT covered | database 3123 K (most volatile consistent convention) | calibrated on Gan 129 W only; best fit ~3600 K clipped to the 3315-3560 K bracket; physically defensible via measured Al activities (gamma_Al ~ 0.05); 5/5 falsifiable replay legs pass; replaces the plume sink | **keep - the recipe's one calibrated scalar** |
| plume `aplume/rfplume/acondens` | none | - | acted as a beam-power sink compensating the over-volatile anchor (energy audits); point-valid at one anchor, harmful elsewhere; acondens returns no energy at either setting | **retired from all recipes** (knobs remain implemented) |
| sigma table extended to 3123 K (the pre-calibration Tv; clamped above) | Ti64 only | table clamped at last Mills datum | depth-inert everywhere tested (IN718 A/B +1 cell width; SS316 A/B inert) | **empirically inert; kept (Ti64) / dropped (IN718, SS316) for width bookkeeping only** |
| `rsize = 1.0` | all | (profile default) | ablation showed self-compensation | **keep default** |
| Gaussian beam (`ilbmhtb=2`) | all | uniform disc | matches the nominal Gaussian of both instruments | **keep** |

**Validation status (audited by adversarial review, 2026-07-30).** Stated at
the strength the evidence supports, not at the strength the pass counts
suggest:

- **Ti64 — weakly validated.** Two independent quantitative depth points
  (156 W −5%, 206 W −7%, inside ±15%) plus threshold dwell behaviour, on a
  recipe carrying **one calibrated scalar** (tv1 = 3520 K, fitted on the Gan
  129 W condition). One diagnostic **fails**: absorbed fraction 44.3% against
  NIST's directly measured 48.5% gross at a 61.25 µm spot. A measured-
  absorptance alternative was tested and **rejected** by pre-registered rule —
  it improved agreement with Gan's η (a ray-tracing model output, not a
  measurement) while degrading agreement with the NIST integrating sphere from
  −6.5 to −11.9 points.
- **IN718 — first contact only.** The six conditions in the table above were
  consumed selecting the recipe arm; the one genuinely independent blind test
  (the 49 µm keyhole condition) **failed at −25.2%**. Read the table as
  in-sample consistency, not validation.
- **Onset: the model enters the keyhole regime early, and no depth number is
  quoted.** At Simonds' actual 2 ms exposure both recipe arms tested keyhole at
  105 W (aspect 1.35-1.59, stable across every scoring window) where the
  experiment reports depression-only until ~120 W. The classification is robust;
  the depth is not, because the cavity is still growing at laser-off
  (+0.4-0.5 um/frame), so no onset *power* or depth is claimed. An earlier
  version of this notebook reported that a recipe change repaired this — that
  claim was a windowing artifact and is retracted.

**Where it is not.**

- **Deep-keyhole IN718:** the blind 49 um test fails at -25.2 % (best arm).
  Out of scope pending the multi-species saturation-curve work.
- **SS316:** under-drills (-37 % best arm); Cr+Mn Raoult correction identified
  (est. x1.4-1.5 recoil), untested. Out of the validated set.
- **Ti64 at spots >= 55 um:** out of scope; the final recipe is the closest
  arm yet at 61.25 um (absorptance 44.3 vs 48.5 %, depth 54 vs 82 +- 20) but
  not in band. No absorptance measurement exists inside the validated spot
  range - the recipe's energy input is validated in-scope only through depth.
- **Onset position:** reaches aspect ~1 by ~105 W at half the measured
  exposure (section 5) - errs on the over-drilling side near threshold.
- **High scan speed:** fusion width +24 % at 1.2 m/s in three independent
  physics versions (identical 140 um) - a standing model limit.
- **Resolution floor:** each case's cell size quantises every length to +-1 cell (10 um for most cases here, 6/25/50 um for some -- `cell_um(z)` reports it). A length spanning fewer than ~5 cells is at its quantization floor and carries no information: the tell is a standard deviation of exactly zero over the steady window. "exact"
  agreement means "within one cell" everywhere in this notebook.
- **Single-track, bare-plate scope:** no powder layer, no multi-track effects,
  no spatter physics; the validation transfers to the sweeps because they run
  the same configuration, not because these effects are small in general.
- **Sweep policy:** resolved - no material, real or synthetic, carries a
  plume; synthetic sweeps must use the corrected saturation curve (per-material
  tvexp + the calibrated Ti64 anchor for Ti64-derived vapour identities).